# Shiu v630: полная серия в Google Colab

**Основная серия: CPU. GPU здесь только отдельная экспериментальная проба.** Блокнот начинает новую серию: старые SQLite не ищутся и не импортируются. Полный план состоит из 8 частот × 201 условие (контроль и 200 лезий) × 30 seed = **48 240 повторов**. Лезия обнуляет исходящие веса. Это модельные повторы, а не животные.

**Порядок работы:** (1) подключить Диск, (2) скопировать исходные файлы в указанную папку, (3) подготовить окружение, (4) проверить данные, (5) отдельно разрешить запуск CPU. Изменение параметров и повторный запуск ячейки после обрыва продолжает только сохранённую CPU-серию. Ячейки GPU и экспорта необязательны и по умолчанию ничего не рассчитывают.

Бесплатный Colab может прерывать сеанс, ограничивать время, память, GPU и квоты. Скорость и наличие видеокарты не гарантированы. На Google Диске должны быть свободное место и работающий доступ к записи. Держите открытой **только одну** сессию этого набора: при обнаружении расходящихся веток блокнот остановится, а не объединит их.

Файлы и код проверяются по контрольным суммам. Исходный топ был создан в другой среде; его исходные версии сохраняются и проверяются отдельно. Если между сеансами изменится версия Python, NumPy, Brian2 или протокол CPU, автоматическое смешивание блокируется. Снимки на Диске не заменяют отдельную копию на другом носителе.

In [ ]:
#@title 1. Подключить Google Диск и создать папки для входных файлов
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Fly_Shiu_Colab')
INPUT = ROOT / '01_Исходные_файлы'
(INPUT / 'top200').mkdir(parents=True, exist_ok=True)
print('Папка для исходных данных:', INPUT)
print('Папка полного топ-200:', INPUT / 'top200')

## 2. Скопируйте исходные файлы на Диск

Через Google Диск поместите **копии** файлов в созданную папку:

```text
Мой диск/Fly_Shiu_Colab/01_Исходные_файлы/
  2023_03_23_connectivity_630_final.parquet
  2023_03_23_completeness_630_final.csv
  top200/
    summary.json
    checkpoint.json
    top200.csv
    counts_seed_00.npy ... counts_seed_29.npy
```

Внутрь `top200` положите именно содержимое полной папки топа, не ещё одну вложенную папку. Можно скопировать её из общего набора приложения (`01_top200`) или исходного завершённого прогона. Старые SQLite не нужны. Дождитесь полной загрузки файлов на Google Диск.

In [ ]:
#@title 3. Подготовить изолированное CPU-окружение
import sys, subprocess, os, base64, gzip, hashlib
from pathlib import Path
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Этот блокнот подготовлен для Python 3.12 в Colab. Версия изменилась; не смешивайте результаты, нужна проверка окружения.')
CPU_PY = Path('/content/fly-shiu-cpu/bin/python')
if not CPU_PY.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(CPU_PY.parents[1])], check=True)
subprocess.run([str(CPU_PY), '-m', 'pip', 'install', '-q', 'numpy==1.26.4', 'pandas==2.2.3', 'pyarrow==18.1.0', 'brian2==2.9.0', 'psutil==6.1.1'], check=True)
print('CPU-окружение готово.')

In [ ]:
#@title 3б. Развернуть проверенный код из самого блокнота
BUNDLE = {'25_shiu_v630_control.py': {'sha256': 'b5f200a3e9472c9aa6318acab8b5fb928d23af3b458c2d43fc29909241e95711', 'gzip_base64': 'H4sIAAAAAAACA507a2/bSJLf/St6vcCRSmSGkqyXsxrAEysT4RzbsJ3MDbwCQVEti4lEakjKjzEMTJKdnVvM3OT2AezhgN3F4u7b4QBPEm+cxM4A+wukv7C/5KqqmxRJSc7M5WGLze7q6up6V2lxcXH0l/GXo/ejF6Nz+P12dMZGF2z0A419P7oafzN6w2D4PXx8SoPvxt8yf7hveuyT7Q229BG7v1FloytceDZ6MX46ej/+cvwEllyMv2KjVzB4hWDgYadrD9lBqaBrCwuj347OR69hCcB8OX42uoTHK1w8esEAyjejH3ArGHoOeD0dPxk/Z/QWdkdsQgwnWI9/BTu9gSUweJstLfl2f9gzA85Gr+HFD+NnhNMZQmMw59XoYnS1AD/e4XGvYPgJnTIF/3uJOqEjD4efNDb6EyxGgLjqDdvybMfigeuwOzsPCaDY+CUQ49nob2JjPIa2MPpfhDNNVKAB4nWOKGUliHdIJaDkcybRRIq+p31fC8rCEwy+AMSeSJwv4BXQhUBdwdM5u2vvDz3OctrC4uLiws/ZlusFtuv4zGybg4C3Wcdz+2zNc3130LV7ptHyTNsx+m6b927RT21wzNT7jV22blvc8XlGAzB33MGxZ+93A6ZaGZbX8wW2Bcvtgbhp02mzDdty2c7A7vW4hxtzr2/7PuzNbJ91ucdbx2zfMx1AIgtYcM7cDrO6prfPsyxwAcYxG3DPhwVuKwCkbGefmcyCnQEczA26AMh3O8GhCSfELU3fdy3bxGO1XWvY505g4mlZx+5xn6lBl7PFHbliMUPbtLnZA3i2w/Bt+JId2kHXHQbM437g2RZCycIkqzdsIx7h657dt+UeuJxI4gM4ADz04RyIbZYBHe0O/uZ0uMGw1bP9bpa1bQTeGgYw6OMgETiLZ7nlesznPUQNYNiAPZ14giHNwn0GSNhAksrHkcMuXGniNDbi1Bl6DmzLaVXbBdLRro+4FeAILui4vZ57iAe0XKdtE6eswNJdeGe23ANOBxIX77gB4CuwwJsYTC5YvvK7Zq/HWlzSDTYGKpuJM3mIgR8AF9hmjw1C5kydFTlu916d7Wze3f10dbvOGjtsa3vzYWOtvsYWV3fgeTHLPm3s3tt8sMtgxvbqxu5nbPMuW934jP1zY2Mty+r/srVd39lhm9sArHF/a71Rh9HGxp31B2uNjU/Yx7ByYxO4vAG8DmB3N2lLCaxR30Fw9+vbd+7B4+rHjfXG7mdZAHW3sbuBcO9ubrNVtrW6vdu482B9dZttPdje2typAwprAHijsXF3G/ap369v7GqwL4yx+kN4YDv3VtfXcTOAtvoAzrCNWLI7m1ufbTc+ubfL7m2ur9Vh8OM6YLf68XpdbAZHu7O+2rifZWur91c/qdOqTYCDJ8SJAkf26b06DuKeq/Dvzm5jcwMPc2dzY3cbHrNw1u3daPGnjZ16lq1uN3aQLHe3N+/jMZGwsGaTwMDKjbqAg0RP3g1MwecHO/UIJFurr64DtB1cLA4aTtcW7D5eOwO5H5iez8Nnyz8IP+5b4aeu6Xd7dit8fAQcH37um0E3/Oz64ScfhdMHbpyMHEcfA7sfbRd4psVbpvV4gRTiAKDBRky+3ULgCzubD7bv1FmNKd0gGPgrt27tgxoYtjTL7d9C7emD7rs1R5cGoOFuVXOtdjvHy22rk6sWOgVetsyiWakWCsvVcsfinVKhVa6WdAUo9AkIdavntpA/lnLCpLwFLX8OBund+LfSALwMzckl/L0A63IW2o+4RRDGTVj00Fqy8RO0q2jFLkaXDAwVLLoQ1hj+X2oLIDJwyyALNXaigEJwOOlBX1lhSqW13K4WC7mWpZtWq22WrbxeXjZ5sVzmJdOq8nzHMnOlkpJdYLE/isOHngTR4uXlYke3uF5cLuiVdiefMy3LKlrLrUKlUjDNQmW5WulUldOF9frdXcBiTynn9WK5WF3WS/ChUq0UKko2NljI5ZbLueXkoF4p6MVcKZcYzFdhj1yhmFyuV0uVQmpQL1X0XL6STwzmyoVqpainBuFvOZ9L7g62sVoqLad2h0vO5fRlpbkgRDx1tuVqqVCulFLI5QuFai41WC6WKlNUqKBBXi6mMNZ1vVxKzdTL1XIuV0gO5ivAkZVqEuNcGW6pVE4NAgmAtolB/FctVZODBWDxfLlQStGrUikW08hXc9VKqVhIEbGq68vLqeXlagGRSiG/nC8CiORgIbdcgItNLs+XyvBZT24EGBXyhdTZq7lyqVRKLc9VykVc3lxALxhFpMc7ATL2ZFJJz+eq+RLcBFPIaiZfLxeLeUAjDxx+Z3P9wf2NHWKELXA6jh1wz2zLaKzh2i3XD9JD8UlOmx9NzwtH70jRPbCDY3yuH1notLhe6okdscTU5sLCQpt3UDX7BilnNbNC4jwAPEOFra16++RqbeGTp7a5b3n2ADVFzTDADzOMjFikme22YcrZqjJx1AENk1RLTfEBEW4E3hAHu7w3qCmjv5CH+26i1ELX/yIRL6D6Q1/6AwEMvKHwQ/rXozfKfPTaiIXVdcGb8Wt74oKjq2yCA8c75rAX1MSbeXDAuf18yB3reKn7BSwPjge81um5ZjABkNN1TZ+33lkCL9Hs+eFa24mvnJApdu7zRNgy/gaMwnuIfnIUQ7Di/CNz3p65jUTO4wF4kbCQrh6X+mpGsontDIaBga62yp2DLONHAbizcKuSaRyzz4FvXF+D1zZYAW2fB2KqomQ0dIUHqtjG7qAPSStWIvsBxtTn7KHZG/K657me2lFOYPWptHfICd8DCSA0ggf29/8Z/SEMQMmyYej3VoZkMjr6+zstpAQYeMAN7byKuybQwJea7YujZdBpjUZxrgYuM/B9Bo7V9jEwUNMnvw77FUA/xIluByLFpzDwQ9pcn0RQYc0J7n6qJC+FnBS6ig648NwbQGAaqDgs8WjxDkgXnJNwR99I0hs8dRiV3pUGT/liSb4CFyf5KgeYk19yIsABHMO3v+Cnv9QVIIEF3o6aEWuRGGIvd8AdVfFaSgaCNNaZkAUggIANnccYG9gB0LFn9sGfWGEdzeNmW82zGyyn55flLwjaWsAtKwmvArDShoM2aBKVYGVuI9rJIVpgdmCHGeeHi1aTp8myyXMfHUXD8TPsZzWmEozJtOgxmnXtnY/+W14neWaXkhPfUZYDHTXkY/TxLjHr8TWIL00YP5995ScKDoJRAeERFw2y1DoOODpY6QMpIYaJd+Fgyk8TPICQgbZdftS29yESVhE+kNZABjCQGWAGkjo241Ty4IHZs5H+BukFX0UHMsuk/weRJyl60/PM45Bki5iRkrr8DOnxw1R6Ct5gbuMt2ANKF6EFoAQOPYHKA6LBWhSh56PL26yxlswdgUCUlrOhDSAlrGFihJhA+PrOsD84Ri51BvHhAQS6MAj/Bu3E+DGcwT2kF+aMF6gpQfsHNOFzmtDxhCYctInFDQh11IguQ59bbg+sjY72hRSxgqrR2Qe9TOIFH2vKMOgsVcBC7UuGsNs+QCTImt1zrb2VLNObqFTTihVmarw/CI5RkeGD7TsmaC/TOZ4oNxzHtZ1hrweRldVVPWUvt1Rt7unw4waoa4jjxXSc2h4OeraFiZcQ0DVCoIz+PZksRC33BC7vgu7xrchzvZMqkcZeEQv8DW4WLTxoxyuZgQvDGmnvMAX4NwqDzmOvgAkEdyAQDIYuGXIIzf9+/OvReWgFAu94grfdNnoQOqJHhtJ1kCFddUB6Co4cuPhWzTSjBcA4Le7hNThgWH1ibXUPVXByLYGNLhfmElMKDPiRxQcBUzcPuAfceUgUy8aoR/oTZl1HXzxvMh8r4sSIyCRLIZGBRoJ4oIzGvxEklupIiAtQR2QJYVvatee6j4foAp4crDCbjmZnxek4EIF7qHXlOUEd4BLKDDn7SMwjWnCEsymsu8lECHSTEUHBmdYO8DDgWGSQX48EQzpy22bIxxLkT2I04gXMXcvsL+Wv31N29hn9fAoLno2/w0Txizi9zsFvBwzR+dMeubajyt2loQvMVo8k+nNtS4j7XfQVUOVJA2t1eR9tLM3UxKNBCiI8D4oNuUjyuGIOeRj+xOfQkGlQZkGtBhyMvCrndWzeawvfheYIliOAAEyGFz9VLOVhhIeVkESQMUZO9TmI0DMyWChqKE0ohaCFL4TPHnlgcZEDlmusZT+gz0OptNyhE0Sk6/PABLMChBn2DSCfH9JPTqsxHanVA39DCmSGfYQSadtOx1WFuBXyGa1vHv0EapyJKsQZWA5gHDjmGROVBRwM2WmaQk9o9r+KDA2m+aOcDeZY3pB9u4ofFg4St4wTDAfkuAH6pLtVOm1Sh8ChbrOBSyrrg/MOOaWqPzxVqKWez9OoyJ0mgDZch4sL63LrMW/DkL4Q+ngtNCKUZadbRE/PoDGQc/pNTkqtqOuGruuYOO8N+45fm2ZcKSowAVig1zPEvZMf6fZwBwKnSQApT3HGPf9uVvEKPrwAD+L1+DlWgCiqwnpV6KiTyqTLQx74TYzLwanDmV+Hflt4r4KRe0inEwprEmiGcusa5H2oX3DPNTDVbrhO77h2F4K/2QJ9OiFLlj0C4LjH3ozkQDMbvppOEcRMGPhl9j78PIxAJTMCIZRYCmHG2HQiIXZ5KIJ4gTb7BdOFwyEGHqUHbPZRLSHIydnpl4mLnkycQZE1pYmuvFy5Zzcz0/NTGZfEgkfNeZs57Be15CGcudpnHgwVrwC3y2XYP7HoaSmXOP8hjtG7G8zJfJjNf4xyf4/2T1YryRKKkmlcS5+R8xT3I2QdVWSOaT6qtre01Ze06mkoG0K1n8VlYq6+I7XjoBIJ1clNKTIJxR/TSXty4gosa8I6W2jD9PCjSPml3xxOJDXcsjZrTzQ2csLPasLuXGtJ/oT0fUb18HcU33+bDG1k3uoF0EaYSXr3VhgUUkNoK7R06AdeKERf0s8Cx0T4RzAkPmCeCjCGZ6nYFaARPA2wdqggZfDBpaWSIDAgP4VBXJihAz3EVTS7qCDanAq2IqdldL+AMc7bsSDuvygN84IY4SoqYcvbF2FaVFqeKF9Rvh9/JXI40sOIxW6RTyE9WYyj0a6+CCse1HUgw2fa+Swd27U823Ty6EK38iIjksc0gAchMmhbLoMkGAQidXwNcxn73NECLEqjsVNIPSvhLJkcsyDeeqy1cYKu5UAe4VXfl3kliMhI6SsHhg70XSrmb+Drh0D1A8OjS0iOUTy/tFyUQwkuh8AcIvaWBzPyOk2AiFEJzCEMFCfPhtexcIom4PrCSvDPERGlfXCLUFUBIbYEbvtNtp9htxgBZivswO1BADJ0etz3gdkgpLRQp2d+6bT3xcqlfZxuDq+dDCjAewxmnbYgGIS4sBgQ2qBY9xPPHQ5U1ODIV3vEzs1MFtHESnnQdds1pWc73PQwI9kFBd51ezB2ABqVyJSmTUIDcp8HOLfGiMy3GQYf+o3+QwA2QbSmIKmyZFb9gWnxmrixTIixdkDpIhzboyts3qZhgiY+4kkncwTxhcUDAyJOvIOWxAdXB+ZTDgRlTpIPdncdAxiupuyjwjlUMOvaM49rOa0i7i8TQtNkFU61a4JoKNJggB+FjyjVzcl0NOPiTSjhTWBPXcuXJXvFcgcUjKE6BvlGN1rktMn1ExGajHRQFHD+ngAstU5z76CZCJAFBQauTXObkSNoo8shHEGCNNGcQJY9etuUJJUcPvE7AZhmDgagrlV4swWPvus0ML2kThajEAEJNzAzjpFoLa6qEOI9UFiCHDVBiXxR7PRQ+hB917GBOeTVDezH/L4YEdfnAU97bemUST4JDl3vccjd9CDmwiVkQ3hZdgNPEOkYVJoqaU5xX6iHMMyx+1zDH2oCuOYNHTUHvnEo5hOWPTmVEEjY/CQMEHGCTDN+LqMkcQSf8QMO3prIPWVZaxgw9DcpLMa+Dj8Q6h97OmihDx6XecBFw0dn6FDRRpNmyQdVSJqOiggrZBSygo2MvlM18DLgCtBmOYEqiaIR3DQvAQKy2tJsgk5I60Cqv/y/YMrKzSyg4rBDn2D6AujeNFR0FomPYzyMPHdoQiwibwBWgnJz8HbpOcsKMg8Cgh1eaIwvkC0kr8BdiLS7haFBD2U9YfYFmaV17pu2E6basBQjqnJRrU6sGzrTRRfl7vpnxvaDDWOtsa3MLb/A0njQh7SIL4y1ulGrn8ZGfwz77cBwUxaO0tzC27mQ2bkoMMpSW1bNP8ZUY5t73sQrlIcVNtodBiDhYXUGkILwVYxp/cdtGxPfHgferO16Q45lJ3CHDPcxPYa0I+tPvAkRypDaD1CNiKyqgs4IDvkRuysUZgIttaiZkBQhKtxOj5gI1uEEWHcA5Fbx8wymQrcCQSeqc7OqktRFCf7N+CvKIcjewxmRadjseAYP2FF5e7qiGTpEoXv5K8pYXEiXU/QOYkIjag6cMqWKOQQD7Bm+O/QsPIDofYEzUysdigYyvQXKwCS/cW+GLVZijZKYe6OPV9R/+QrzjLH0bNSxIhtQIO7+TjZNstF/kLsnHMHXIj0nEpQJck0SvencZyJiOddm+g1UZWbUjYobfS1d0O8Ydd0ANwPhLjABEO2e0wXFX9BNTdpiBcfPyr9ipgfzjJQxA1rcJmwphyDSSsKdjZo7Lxklst+SOJ3LNtA4BBg7p3jhW0ZO82vKNn0tiE3FkCeRaz3pCSKnmhpewUFmFJ1Q8fQV+exniSL6+BkJOEP8RYsuUoCR930lkuxiE1o2uphH3D/jHVEZH8/zjLhgVp2ampKexZtcYdIlnByNiYxUnwOW7yYMgux/IXP/L+n45zLEvaR25lj37pWsrj7HV3D0PxPDvaQA9yzWIkXUeUUxcRiOiGbkeHVOxGsiV5QMRl5TDvAJkSYOIcJ4dlhzJtLRF4KggD4Q4l1SaonVgMnmkVm2F+NVvQpLICJ1iVifEY8BXUkZwOCzZJwGJw/DMiDO7xD774lhMC5N5LreMyJ3VIERncgkvxiywvCFRBlZ8bCmS6l4nyQ4iQApvksRGyPtSUzJhjwR6QgpDVQkOkfKxCl+MYtoQGo4YZiQfUGTRX52SqOikD0fvSIiX8Kp4/XH8b/BjBfyKO9kl3rIQ9/O7Mk+m3czf02R+lwi84os5lNxJxjmx9rJUxsAa74Iu9gvRIs3mdmvQ5xkn3g8rRxlpq85StT0H1MMIhSP5fdDWUuwlhCWM3mZcMOa0jydUUkT7kQOc2NkUx1DNLTgQDGsbejRa5J1eMjfuFHIg/+aXCOn90VTRscGH4qT9dXijn4mBpZNvSXouv7BpNkfJNeHvPUNy/3jy98Xp74TkI3KzkSgM6bqWdygyUa/H/+aNPd0QuwNKbVkgthBZy3WSEP+1s69xgNj0nO7Q1UoWVaOrZYl5HkANuoPtsPFln8QWyhdu/i3QVDdYcEi5aZMSjloKJEe13koWNbQyM8Dt7ob88fwD3aUcH9Ga2m8d4UKaNl412j8rRzNnMb8RnT09hTRcqA0ZecK96dmSP8m6mXoi1qEAe4dB28SfB5cfPKYH69IEHvwubmX6n5oYqTMjwaAPn6XAngOZmUnIxAjhF20WPPo+2oMWykXWEn/CWhNqqM/Oed7Ht0N3lyU9hUKEl2y76hmNbPXWNrhiftJj/SVHjb6K1XI0Gl6I1yVlKKXJWZGSv0bWP2WEslUlSfeeknqEXuYz7Ufx5mxpDS2t6HrF88xzi/pUC51Llti0Ig5o+ubVxKRQSbRzET3j6KHd7+nJpg7yyRDqxFHR0AzzVRPE8YrCCfepxQvo/jBpLVHfE72KSW4VvQEgcueGI0agpopXprXvDSjceNX1/QzkUOQbuM5X2EneK5TJTMllUh9g3LrQvxigj+dKIyR1pCZbjGBHprXZQcTIT8FfIbIpodJMHnBczJhGDI6VbkE/I+ESCd4I0nWeVno8I+0cFG+LM5XDvKTZ4L2UxP2cMbNCamB6/pj8utvMxQ6CfBKwp6xE+dm7vTWSWKX0yyY74T3fhb70pp0AV/G2tcEI8yXs1ieVqSNZlQaokvITlvwbMxVuMmceYD3FMKfuMmBibmpeeJ8YVpRrEpCoy5GVWYhbomrR7JJ0ghjKjscD+c0aaGYH2Iyu6aIFkg/8LjZn767Q88WLYoAVVuzreBTGlDFfNQsvNem1F8tPBtwo2zbnc61ZWcly9LZqmZmDhoa/epys439rddOQplThbkk+zTRheIRmdc9DG0gygm8RD01I+92SouxqQ5TbILLk6ZOqguZzMHLVdzHSmIOkYJAojwBsL1p6jSnN0oKnqBdEso0PT8MRuIruEuqNtnTvSJXwKW0h57QRX0czlEfhNIOxKOu5bC3k1QHNnDmNcM44B72AxvGj9B14vDcdERmc/LFKA0H1Qm5MhOemTs7RpfMj93bb0/BwrTfQXxr5AtU9PGhj0DzkOrF/pIfsZnAbe5ucdTD7RJjP3U//NIYqA5M8onqYFRPNfoPjQH3DF9Uf+gS8+UiXirWd+Q1a5XTmcwiEm1N+kbILJWD10RTteAoUJIMh5qGOPZH5h3T2ao3E29pEiRG3+GeV3jrYJg7yTedIZCz8Vf0Re8odQHWP9Lrp7dj6eMw6E4EUuHsuOo/paBKm4eFgvnCVFow6vil7S5Fr/3FdOJUo2T4tPFNyPa00iZyhyakI5qYI2tKSkNaoVNRKpeDpLWbp+RCMNktB/G5fDutrcTBs0x+3fssNnmGUhKzb8dTVmJuUvefAjhtMTODfW4i/yBBrv1mybyk3HW5Z8rzxFKI4++SSUSRhnkV7jpJU8j4463Me5xP4poY11AW8UMZNIw9/GG/b3rHGn6NVEvdeszex6QsI4ydEfAjCEt+6chmUKJXBvtDYWjKAYj5uck+unmmLKoxGJbbH/R4wJVYigB9UtlYJ7uF6/QLi3RTvcEzoHN05pXb0TvxjK86ygn2/qkAIgPWBb0Mw8Avm8DAqZIKyqIvzGogIBCmGjAJvx6QqunM9v7kKXILs/yr+LVc51jNdqToMtugiGVMnWWh68Qdf4iNRr5l26J6m6VqtBPU8lmMwt1DOLMTL+xGnvR/znSbTyQNFwV9F5unlFR+Ov56/NvxU3gfHWoxfqjF0xlOsSx5IWUWFhbAMIUXQBGIYWCtzzCUFVnb9zV+ZAeqqABmFv4PqjLi6c1DAAA='}, '27_shiu_top200_v630.py': {'sha256': 'e630558dbb35e34d5af45bde417e1b91d36e2c6a7c4417e87be27fa2565b2f08', 'gzip_base64': 'H4sIAAAAAAACA51aXW/bVpq+968447kg1TKMbCdtIo8KuLHSCOvYhu2kU7gCQVFHFmuJ1JCUHTcwMEmm012k0+zsXiywwEx3Z/d2ATeNGyeNnb8g/aN53vccUqQkp90JWos8H+/3eb8O5+fnh9+Nfj+8GH4/PB89Hb4Soz+NHg9Phm8Ffi6Gb68slstieD48ERh8TmOj348eDV9j7pU4+GAJk2+x/0zwun8f/dESAHU6fCmGP9HP8Gz4yp6bG343vBCjJ8M32PfT6GvAOh+ejb5VSH4afUMAFaSL4fPhKX5fD09sceVK7PcGXTeRAsifDt/SahD6DEsei+ELbGIKaJeijZ6WBe0e/jh8QRzNgdyL0VeYAtLRn4fnAq+PsfeUEZ8PLwhPJONBT15pR2FPwXsJOog0jUqR9oLx/wjSH4++IbEQhyeK4tE/E3AtRb3+B/x/bs8N/4+I4/XEx6PRM3Hb3xtEUiysCshOTeS5Px1D+h4iJKwE7QyiO2Xxn5GqaN1LSBV8kNJGj+25+fn5uV+LzTBK/DCIhdty+4lsCeZrNQrjsN/xu67TjFw/cHphS3av8l+7fyTMu/UdseZ7MohlyQaYW2H/KPL3OokwvRI0vLgkNrHd74vtjj8QbtAS674Xiu2+3+3KiBDLqOfHMXALPxYdGcnmkdiL3ABEWKBCShG2hddxoz1piSQEjCPRl1GMDWEzAVF+sCdc4QEzwGFt0gGgOGwnhy7kRSjdOA493yW2WqEHrQWJS9yKtt+VsTCTjhTz23rHfInRtKTbBTw/EDSbTopDP+mEg0RA+0nkewTFwiKvO2gRHel01+/5GgdtZ5HEAAfAgxh8ELWWgBz9Nv1KZq4/aHb9uGOJlk/Am4MEgzENsoAt4uVqGIlYdok0wPBBPXM8ppBXEZ4+CTbRoopp5LADlRa48Ymm9iAKgFbyrlYI0THWL6SX0AhtaIfdbnhIDHph0PLZUirYuoM5txkeSGZIKT4IE9CrqCBN9McK1lNxx+12RVNquQExpOwWeIqIgjiBFfhuV/RT45zglSxu505NbG/c3vl0Zasm6ttic2vjfn21tirmV7bxPm+JT+s7dzbu7Qis2FpZ3/lMbNwWK+ufiX+qr69aovbbza3a9rbY2AKw+t3NtXoNo/X1W2v3Vuvrn4iPsXN9A1Zeh60D7M4Go9TA6rVtAne3tnXrDl5XPq6v1Xc+swDqdn1nneDe3tgSK2JzZWunfuve2sqW2Ly3tbmxXQMJqwC8Xl+/vQU8tbu19R0beDEmavfxIrbvrKytETJAW7kHHraISnFrY/Ozrfond3bEnY211RoGP66BupWP12oKGVi7tbZSv2uJ1ZW7K5/UeNcG4BCHtFDRKD69U6NBwrmC/27t1DfWiZlbG+s7W3i1wOvWTrb50/p2zRIrW/VtEsvtrY27xCYJFns2GAx2rtcUHBJ6UTdYQu/3tmsZSLFaW1kDtG3arBhNl9tzfo/ULnDu+24Uy/R9z0ufOm7c6frN9FX9YMAeJH43Hf0Cxp8+h3H6FB9lj4nfy2AnkevJpuvtz7H367sJIdCQxSZe5+Y+XtmuOVDN4vUPRFUYzettxDF3Sd689uGid9N1P1hauOF6bvMGZpo3F2+0Fpfc9lLz2vUb3mLr2lLbW7x5s3xz8dqCvHn9w4UFY25uriXbzAx5I5OQlipzAv86QKCZtHFmgNEs8QR5GabODvsyMI2oaZTg5ERb7aN/bRwhrzMI9uls+YmMzK7ba7bcimjbkXRb5uJ7C+XFa/wHDq9pGKXxZkZuD/otuEyToSi8kUzgKzDVkQ9a/h5cIAhS9LtJ2PM9h8TNLFjiwO0OpAaa9PrghSkm2p3A7SlObXoS7wvDxhIjxx1eNXOHhiVk4IXkX6vGIGlfuTHFLaG1W4Ne32SsiBu0J0a8dNzY8/3qbbdLDtQPWvD91UWL3E14CDoCNVUaC85udwdxxywtw17sdnwUeCbGoJogNEtqHSYi2e/CWEzQaTFjqSCabiwpTA6gS809TYN9sh/TcQiU45RsWDVoEVeFsXjdiREdHUqOHDjYJAopuhpK5u5hKjrSm9M8SmRsljICmsbn0eeBQSrEj6LPb0/aDaCU8loTv6qKnCWPJYkoH0txn6RYi6IwMi+jDlkIMpk3lHeoFOkMGccZ/dCYSraQ4jwZ/jB6grRNZzyPKIdbzjJDrEUOMp2knCDj+Xr0ZwJygcfHnP1RpkMZ4rmt2Yz70oNsigffplGHzq+SdDf0OA6bBjNA6jFSjREQpappMGpcASKQJv2BUdCP3Q3dloxs+QCotK7VT+GcqCFtFwjobhfyGwRJbBIVCPNu4iLUStnShqIdTQBDPiILD/r54SYgBIs03lzk8eaijRAZJU6MsCnJYjHSj2Q7tnFc5J4M7ITSpoQcFQM1eAnIcQfdxINo9u0WzZbthfcw0Yu1vUZuL8bwQ+PAKRuVK9cXefa+JTAQxcnkUNLByLXr2Uji9JqRUVksK6A04g6MyvXxqxO1PSywFZT4mPHK3xFSo3VwlYkygVxcEQdwD3slnBOGKiriIOwivRwESN1iiLoNp52E0VHp86C1p3Ze2aPl7uCdi0EB5mNJGY06aoEcRJRlVElI6/zySRQO+ibOvknK2jX8Vmw04C9BKqVtSPZaVaPrB9KNYFVJB2lhJ+xi7EB8JFgwFqWKMqGRqmDpLYs9Evl7vfs8mZJUNUgmliCPGPdxtKtKD6U8bfYBOwMa32XlNJazKYY6fiX+xmuVxBvq3BwFisfto8Dtx3Aneo8lsgfjUAsPFIWBA6uqGnvi/aogd4zs3z2qLtg3lPZKKVSYXRAgbTT9qhIXthkNS3yRvoawnQYdIqwlt6aGDyWnx0bjvbK9+KE2I2WJoU/q2FV0U0yLfM4W6QDZnAWNXZefAeyG4f6gbzR2sXosoF2/oYVSzhl7isZ2+wg3LRMzm3hFQKkH/UFijjeTqYP59eoC1IbIWEXgJ0B3vrSEYqGq6F+8rhDcL6UuJvChYS3zvr8v76qRsdwjmGHUyseigI8tG2JyGEb747UQnpXCtMR7RHwp8wdwJiZ7FAiZfANgUIZj0x/yEABrR4PAXCiXs9M5triHxwqSclPYG/RtN3ajyD0yNUab53YrEEcrOerLKpb4QfLBtZJNNYDOUGAgMKwWIOAAQao5Gq4wXZZYUithS0RUjiNiSHOpeeaFex4QoGj0ErPgZRWtVopSu1uOlV5HevsAFyQ6J5EP4L25sNSQ3+14QWlCoYHTC/L5scmArhpjyDZNGiUVnBP5IDGnkpUsKDM8Gw7ZxMkIkxD8GByKU7qo8EGRhLrMD6j6QXzP7fHCXr8rscwoUY3oJaV3xe3hf1IrYfSYewSqb3Eyesb9E4Tb19SfmY7VGDoVoz8g3r7CnqdWLm6r4M4x/i0gUXPkDe3F7NPlyXYHdSMIFDc7aOp09C88eJEG78zEHh5nh5sMl2VImVo7pIPO7O/mWG/YyGZ7sZnjHZIloWW7bT9GnuNTmgOgpP1sqqTEG0BewZ40l8qlGSIn3FMCvkTIfwFXr7nFRa2V15CuasIQRghPcN+rqAXqg2VaOE3lwfYm+cgUKV4WbZ0AIqi1DSU3h+cf0t9KebF1bAdpvpjL/7ieaLOBEUvKiFRCaPwcZ+1L7ecRN+XekP3ku2gngqk5Vq0p6qDBdGAoMIeTPI+e8ip0msx2mof3fW+/KydTcbDh2exkiAVs4mfTYHdjsOY8SnDVfOYglUrhuIIj0xO/EeXiwEeCfF/5Fwhgtm7B3NfIWB8RY8TnWz4spObnrPC8UE61UAoCYAXu0niDshwP1QqkkbDZIRhCIo7KSijVmOHqtIfruX6QVRgAlBbK9kq0xy2uTXqLzJaMvcjvUw5cdZxW6KH4UJtst9VyXL3aNMatU0Q5l1tbVSOGS5ZOEg2kcdmuvt8Nk9lbLNGR3X7VGP4t37QlN3GGkgFCOuXs/oJbta8KslvOGdJLbDjhcmDcYr6UnFxfllMVznWrxpiW78glKQfIngnuitqyP4z7ydyEZe2KKUfPWIGPXFffZokT+jgNSgPKrVAgyuDAhzmqQ3d77TNn6966s1rfAh2otW1q8PXNLDKQB8LWsU32I/IC+Y2650sO9QXVQ7YY/gc71ycswzOSJhdlur/ODWDtiUfPbOAlf1CNj2IgR/USwbNou1LlBPUtdZUKUqj+HSR2b7/lR6aqVePqTkTFtXzgx4kT7vOrTv4GvZ4bHXHNQC57EBsVg/IpxEHimDrGGNEGBjMxiGuSm5316yUOv6CUsd2lZMqwCgcU/wyucwBm+Fcw+z1ZyT9+57C4oMZOtO3Fgz03Ep9srS9PX0eMO+4TrXh7BpHElFE5gGGY9IhQbXCb2ajsUgrpuQfSTehtcifvBnEUiN+SW1HazBGqTJPOxMnoK47B53qILfgVGy9L4fnw5egZ1c3aIialQnvazEVs+/2joJnjSQz/jTA+gRgQ7emgjP6owz/huYDBQfqYf00S4RM8+oardkjrJy7mOfS/ZVKzS5ZzinXWbKb/O3ffwtcnLObMz15M+1kMnI6+KvLNMzgfYoHWk8d+gtEXoydIY07SKxN9H/RUaZOOP/cw6O6I77XOMv8EfK+VawIiEERrX/Bp+opIO9G80TUKCDljEpWXoxuWjIlHLHY6oOqy5rWOnDn2WLNvRk9EffVSEf2PNoHz9EQTEHXx85bhkOyeKsgzDgHfo/EN1vBsWTBSuvFKMxJ9mURwnmP4jP+qFO6wWqa4hpnviQOSrJblJSxDkH9hwyPk7J1IUM/VDRUxKoY/cI/nDz8TSy/TcSG5OCMvzUsK/ShW8ZtJSSDhFTCwFwzsDV0H/kSCvGC9qlNBcXk5sxOV8vwJW59roXAHilIB1ZnSl3p8s8gSfcmCO7lMj8oVkXU+py3EOkT0NfN9zpY2eTep4+Az6n7hTH+b2pU6b8wmGe2PTN+FTlIwBDX8NccXT32vU7Wi1zxVlKgLQc5Lz2mDFs5zplIFxjO+5DxlQ1JGmL+VVMXB+BLXaKjUPomOCsk6u33OGviWyKTgVwwFyNd4QMVybsL9fCr+XxzKScV0IsjE2QFyBnvKDosqnfwFcf4e9YzNHWmtXpVLI/LZ2yWV4rvadFyauzEVkoXm8DjtV8Fz13AHSSeMnDgcRJ40GnqDvb1xb+tWLZdABkE65VOPghudKlfYvlO/54yvYbYp9lKS8ruBTIzScr7Fden29dq9rXSrFx/kmNdJib7/V5XhM/KgfNgRkvggKd86efDyZaVtczpCffZcDsG3DihbpGo96l4S3fsZFaa2jWRCRooImqXQqhmasSKt8I+n5cxsxyxghbBgnagS1OjufmPXQB3pNLth00GhsYAtqDQOuFjdtw6yTlTtt5sQeG01rU1/3lj/N5XFVL1Eqe63lOo+uiyZsX+ZSp4rV02G/YKj4SPCdqZKesSLy7VAXbTURA7crk/3QI4SGgs+1ycsnNtS4fppXx6pnJPEtGsWFGpp/ZmZAq1UX42i7OKkSjCo051wi1sCtqymGpJHjcJyKNCME6x2Yv9LaanHHnWeHMD+VdVU+3cNvkcxGlb6nq5BzVWZ8tyzqkOtwdw9CDkd8s0V8ZBIPjZmHHESrepLGY3qw7H5TreYiwdAzUXhIdGMTDjJMmmHNlTGDdHjnG2oHhPwxEDZc43KgpUD6+heQEULs4ARpq+nG9kxm9qQkp9fPCPsGaoCLWzPXU5NXIy985pL3XIV7rJmYmRPm+LL3XSBl/4RvGxgVKgSOpARfZlgx/0u9Yt2y8Ss8uCQ6aLtOHqF41jpHUol6BfGZzLsc70TqULGaEcSHjjwjpzOl3QvUraM1iBS6utBe9SRwFDCL2UbWqI0hMwClZaZ9apm+bKskdiopo+TjbHCGZ04XSkYXbQ1quMCzEmbbsayoPukajnbSqVaEdBkO28y3Oci+fTxUgGvynYwFfenVlMjlTfQsrB7gFAqqlWuVrOByqzka5YfnmgGTHQwKf9JB7iuopRClTgnlPtxJkaDQt9PXqguhoJ5NtFPyPvtotyqk+1qxaCVHWFLTDqIaVAqFLQLsUD1XLXXv9CpeOEylbsexYwT/ouwKdJKx6qEpLAzO1owGxnpcDaZSVbG5Oc6uJWHx0UbyX87AC1Od9et8UARbRImbpduIL6UURhP39NN3FAU9qYN53E3eGEiOeWGxFJ5hjGRCeqtSkqz7c2zchch+b7f1OrpAzWl1b+lKXuloDDVYXx/4fjqUlldJmc9x8vVNSZwTN+7L6intqub5Srp651N6WWBlKiv19mXLKSPP6YwqO9A5PhDkOmvXPL/oOXYxdlvW541o6u8/Iu+7Mj/y3/lASosxfGMI5wZZ+GeYjdOInUVR4E+H/o439bQrGLDt6L1cTyF5R8/JNlBoetb73K3kWvCXO4Yxsb2LvsqnKMZ/n5G1KGV44hz+RYKDd2Ek6dM2I6Olip8syFTUMUz9aQPkNalaRYXBn1lhk4QBuQ3TBZOaWYeMR3Zu7KNMB3cdOgKmMM5wWQQuxO3z5yT3V2/ucubjEaj8QtxcNrw/0aikg3Ccny5+FQrslHdnbKfxozYVAj6l/upX4tigz/XsEHJkfZnzum7Y258cJQ84RuVM25p6VbK6FtuPuaaIqcibXupLlyu7ZaW7arlM6trqDHVV+3p08R2QZGhjaSoYAgflWe4gqglo2qM8l62TLXXTkJOzpCpowipqq/rhF8xr4xV5UPh9JaLSTRWmpnTUPBiPKXfkJP/ZQkMju2/juWo+kdT7bL8F/MPx2iO1RWLbt89y1+1cEM36xHNylt0s8OLD2b7beWjkrBP+7mLMP6cb+KC/Ers71H5d0jf0VSNdzr5wypg2YcRf9HYhic/1C/hoYkT4Ab7yLajEK7Eb+GpJ90gO0OWwVpxYvoCI3aWyo72FI3ZwZE/NwFEy6c4L5H8SwKlpLdLWXzDWihVCiTw+qK6rYI9UJQuDMxAPsM9hvsIpLN84KSDG6vXyhwhUGpNcI2FuiSnl9keyfAGSdhuz/I/iv2FmzcbiptpN3xp0fIOb5SnaNo1FQoQ+cCT/UTU+Ic+K4e5YKwy9w75SToteRGqgUa1bTzkS2UAKKGqo09hHOcYgQ4Dx2nhszDRbsm+E7ZhIz03cbCYHMHEBZs1GR6nYrgmRwdw/aYvNmfkfQ9T8ucVZ/ONY+ry4rjyx5JYQHd4V8V8Hu78sTFFSHaf3KLPE+F8Us6pkDIch+6WHceo6A+7Yls+QHmsbpxLc38HavEYWQM0AAA='}, '28_shiu_lesions_v630.py': {'sha256': 'fdedb0ac3c7b81692b44c6b5f07f2e2142e3af1617db179426de3b250b57dd93', 'gzip_base64': 'H4sIAAAAAAACA8Vce2/cRpL/X5+io8WBZELTIzmOnVG4gWKPY2FlSSfJyQbaAcGZ6ZEYccgJyZGs+AQk8WUfyF6M3TvgFru4W9zh/j3A9lrxO/kKmm90v6puvmZGtrO3uHNiiexHdVV1db266Pn5+dM/jL8cf336/enD8d3xb04fn56I02enJ6eP8PhYHLxzodEUpz+Mv8CI+6f3BNpOH+D5Eb+/wM8Hp8/GdzHh+fib0ycCA+8B4K/Gvxt/5czNnf4nGk5O74/vnJ6MvwKg03v4+3R8hx6/x0IE4S+n3+dL4BFN98TiJTH+UqSjwcBPjpxP0ziyRRYPFxsNp5seKCzGX9KqeLzQmDt9ShPHXzGQZ+Pf4vkb7hv/I1Z8goXu0asjTv+MxYDNcx73KzS/OH08/jZf/wHQ/ILIIWQfaL58syTOnUuDwSj0M4nHYRDG2Ry6v8mJ0DPGX+nVnxJFEygx0gCHtjsFi8ff2lXYaAJ/xndA2lM8aZapJYi7eCFKXjhz8/Pzcz8RG3GSBXGUCr/nDzPZE/0kHoirSZzGw70g9L1O4geRN4h7MjzPP53hkTBvrGyL1aAro1RaDsBciYdHSbC7lwmza4nFxuIFsYHpwVBs7QUj4Uc9sRZ0Y7EFykOZ0MIyGQRpirVFkIo9mcjOkdhN/AhI2MBCShH3RXfPT3Yl7RxgHImhTLCRIu5kQCqIdoUvulgZ4DA22wOgNO5nh34ieUk/TeNu4BNZvbg7Gsgo84la0Q9CmQoz25NifkvPmLd4mZ70Q8ALIkG9eac4DLK9eJSJRKZZEnQJio1B3XDUIzzy7jAYBHoNms4sSQEOgEcp6CBsbQE+Bn36LZm44agTBumeLXoBAe+MMjSm1MgMtomW83EiUhkSaoARAHumuMSQR9E6Q2JsplmVUsvhHra0Rk1AOPVHSYRlJc/qxWAdr/qp7GbUQhP6cRjGh0RgN456AUtKE1O30ed34gPJBKmNj+IM+CosaCeG5QbrrnTPD0PRkZpvWBhc9ms0JYRBmkEKAj8Uw1w4J2glidu+3hJb69e2P17ebImVLbGxuf7RytXWVTG/vIX3eVt8vLJ9ff3mtsCIzeW17U/E+jWxvPaJ+NnK2lVbtH6+sdna2hLrmwC2cmNjdaWF1pW1K6s3r66sfSg+wMy1dUj5CmQdYLfXeUkNbKW1ReButDavXMfr8gcrqyvbn9gAdW1le43gXlvfFMtiY3lze+XKzdXlTbFxc3NjfasFFK4C8NrK2rVNrNO60VrbdrAu2kTrI7yIrevLq6u0GKAt3wQNm4SluLK+8cnmyofXt8X19dWrLTR+0AJ2yx+sttRiIO3K6vLKDVtcXb6x/GGLZ60DDlFIAxWO4uPrLWqkNZfx/5XtlfU1IubK+tr2Jl5t0Lq5XUz+eGWrZYvlzZUtYsu1zfUbRCYxFnPWGQxmrrUUHGJ6fW8whN5vbrUKkOJqa3kV0LZosiI0H+7MBQPadoFzP/STVObvUNr54243f9rz070w6OSv6hcanFEWhHkr6f78eeBne/lznOZP6WdhkMkLxetR0ZMFgwKDLPG7suN39+dYRw4BCkvpRcUGQZ7bXt/wsH/CFYaE4bt48XKv07lwUV54u3fR7799sdOTby9ckguddxd6F96Ri913/Evdt6np8qWOXLzU9xcvvnOxs9hvXDbmrm22/n4LoHYuNux3Gvalhn25Yb/bsBca+LuAv4uN9txWq3WVBkGDZCbU5640LzQsa25urif7Ioz9ngezZ1rNOYE/hLRLqJqeR0rQ8ywHbIZiPG8sXvJSKGtPWUmPDDd0vcHzgn7OawenGDiaBMlJJMB3jjKZmhZehiE4ZHaMXyS/iAwbvyPDspw9easX7EJvmtYbrmaQwob+wLykUnzkhyPZSpI4MY3T37MJ/RK27a4gOwb79xg27StyHr7Hr/v4CSdBGcsfzgFZARt8QmPJ/pKH8FDbPrgAVbsMk/mCHYYnjmExCulQdt263DjU5tEeKxaFcZc1ummAM14i+7BVUVcaNrFAQYFCH4VyEo5qVZAIpkk/rCVe06GdkYkjb2EtNdBUvxTERGZQzxqw3swDPwx6sGa8ofhri45P5gFtvs37I1ObdDYA6/3WwhmNBrDafiqioYZOzS6dDEYkNU017bxR9ZgMS+1wJm9lJmiOyda5xijrn7uMnc0lQ0FzdmVmGtDd2Sg1sNNGvG+QQq/2kgPBfdpfAVenxsDCjsLMsG8fW6oBJtEPGSQ8yZfJzb/BK/uOnb/cS6RHcoiUU/br8e/09it/se4+ni1Z9+AfquEPqEOJk61kDs2PAYUm38tlSjO9A7yjReJ6Z5Hb5S1sPJwR97aRdvfkwDeaC7YByxpJdig8dbSMptrKnUpXarR3DN3dto1IjpIZ41VzfWzBMPUHKyfBMCum6vNoGyRKRSuEy/lgeatFXWiwjeER/JvIaEI1OgfwLIASjgm0pmntNKYXUaQbzc6i43l6vOcBb5JDoxkN681pALFoGuxHGLYB5++zEcTtyNv73GhiE6bg90YJC483SMHDBkYYvYxfGg5YmkrZwzOrxuNZUjpM4izuxqFhiTfcYl9eKlx/pm1nuSCv/JkttFApNcWNUDq22GBO2UoEn5ISGn8jWEbgvD/OIyX2xiE9Dzl80C9VCZyh0MbfKoWGUScUAHGsRSGJFrtuPIqy1KW941MLEevuD+MgyvThtnNC7VBGJmmNHSMAo9rlUU7BItkzFSicOGbhy5V1Fe+T2sl5yAfsOyJPIID7Bqz4rY661ImaVMwY8vXUQcvJy+LMD12IzucS0Yk5SYLdy46GkvpB8Dtvqzl96Jau7ZGfqUhyDgh3WKumgveW2+WBcQL+uJp6wOhDOUVxREuZPPCnDcvJYjaylr0vj9zQH3R6vgia5jniMA/aCYAHvVUQo7aSv4Q1r2W9BxY1X4Oh2GHizQkemakQKvQ+Bts0u6jvCbPqRZVdhcTv5Bq1rWTfh0I5kF6uK/gElGiV2nhyXneUxf2+l5ABwrnkeSXlPHtn4d13223r/Cv09B9YSP7C28wxKehV8f5ZZ4BaH/BZ+pKedC6AusZf01Dw4nf6HD2vCxCC9pwjFKOJws6VuQBYuXgIBkwYOMTUu4YdycMwiKRrgGBo8n6FMEkG3AUA5yoCwk1+NftWMYB3gI18P5BhL/IHMn3DBVv9aB9aLonjzAt6eBpIPyr4ahvMUDgMwb5MvQsNT9u/drOmBWfZP0jC9zhvj2tqhVlGB/H++Jc4iCXdOV8YWnyYusqFZJRrAkud0AVnC2w1r1IcerKp+aHXB53aSIgJJW56muNAB5X4YtNidmDRiZWwFpLYYn4eDBkJW0kZmQQctAWrWeU1jdDMbb/hIpAmb3hfCTT3aH6js348a2zNR5+1CQpy7cRbkwAQ7nKg4QRpN4xTafbhY2UKv9pea21RQMLBsRMZwr8L3QV5bmHR9jtp+WZZrxCBvlHdib/N6bEru4UzeZuYepxvm/ZSd+oM5d0MaAsr+9W2SyWj/Vnmqck+LLuwhd23yYDbGaV+spc7sme5WnA84Ikm8HO6ONsmnG60DOG7pw4OudyVkaPAu9on4QFAyofGw6Z1951e5sKZeBPNg1THTok/SOG8HXgNo3nu4iL3fWTjPUmziZZsDw1vX8wb4Jx0EvJlFDw0+COjebF485J+F92OgpAqp0V+lrpG7+B8L0N8Z2JVcU4ciLfEriXOC4YomuIgDjNhjqJQpin2o48gNYuTI+sXUW9XzTy3S8P90UsHY330p5LyPCrk0wbCBUJr/PhhEo+G03YXWNoDCaen5xqkKv3EsLM92I49aFrXOBA/FcwOSHZK/D4ASsyxJbGLx8abg4+oL8fFNYgVNmvLIcJJV7HdquLkHOjWHd6L9lLRses2yhcAKsYpDquznh5FRNXWUeQPUzgDeryd/zYONasMGy4mpMY1dsVbrjg07J4M/SN3wbms9snK4TnaVTcDrVwwC873p/lbDAFpF0pViV55mA8l5wn1YP0Gu0vJNZJd3bJTAe3qwWEc74+GOHYKZrvtNgqwhNehqye/2XAWL2l5LEaAnBy4krgwlc2J+RNITcGBh5m6O+1SjQec16Nj7XC+qqKle7cm0cbocvt2MKDN29aonLt8Eccfwkz3TPRs4BXx6Uo0HGVmdTKOnmGvuQs2qVi3UCgE7frntiLCVRQsXlSLfGTlMXwUQABZMkjt31DvhXQkOBtJz70GMyAtQjlTRyM7jJP9YhR4ZmtI9puEtVXoI+g0k35QBoB0k0vJJYd+mAqek4wik6IZrRbKM4BYWOtb8shctj96FYcd250JtjL3b6y9iybZh+i1tal5zUkqEuNZOPTgeQXVc4y8fUGzjUQIuBdEE83MBc0RHrTbxYphSAekZjgUPdocwD56FMmatG1a+ytLQDTYjJOtNFRapLOOKH2hEhk9KPLU3HnZbGWIcxiUmTKtNkaRlgDyqWvC/7IRikKtRekokZ6fdoPA3U5G9bzMVDKM8XDYhZTwsWnWRO5LU0muptfrmPEoI7Xp22k8SrpSE9TreJyiQ+95bB0FyKmj05NGGaTRjMo5Tboqp6dBIfxL4/BAmjVflEYR3LJ3pjf35zJzouKz3KE74Xuuu1UPT+fjENrSPRh5evAV7nKj0Dd736s7JwXz8cQlXdUDVbiDMy4QPYN2TQn5V8VwOFmcpjuDnN8LDjLuKz9H5f5KhE7YSwK2E8tVEeOwIW/P1Xy5vJ96oyQwrbeM9ym75SaxYaNBiQx5JLCC9tR8vc88gLR83bHDFIcSzTC31KkPGjhzBhQVHycVKECMMosjOM/GxubyhzeWSZt393Ai41HqXru5umrUpWNCpnTzDDifjYLuvkouIHLqy6y7F0eS80Aq6TdzH/44ebsKeYGk3MdGPFIJO/I6KQ/wHQTsZFI6GJ1Mwv2qYrTVWm1d2RYc0/PNhKATRVqNLjk2W4IC9fcNu5LwsS2Nsx+GpjVJLjk3vAzCnQVy5SsZUm4HkfgfvbTQX0MoO+SPdIB2j/PTD8dfwCX/i0rmVFNCeK2nmVQLnn+gTBLnnJ4XuaUf6gkq6pvkIEd5MxhYOt6TetNjQ6G0p34+BOc8rUJtpdwU7/M87Vn89ekuUfZceIEmsW/H6PpRjzPa5Ej+w23DOIZplDLiIfXJ5Fm8hl3IEYIDQsQ2JxKHgiViysBYU8MgCxhJYAgfoRdnzYM2dTGDRgUjb9YE8i0moObN6oZGx4VBGkR0v9mVJmMPo2zN6FNEVTsb77k04T2XHIRGpZWHTjRXIOWMIZ9B2T/LmghS+wFMdzHSUvjz83uNaQ6ekXx4Hbl/yKm+7yC8X6m81VOKMvnUU1JLp0ErAej4m0kRzvUSb/Mb7o/zGV4fb9gxqvB4Xqqmr/Kk1IOcqJOpJO2zaaXFpg0i5Pi9ngmRKnvrvnZdaaskvWlc2Wwtb7fENl/y5rqNwIjt1s+3xcbmyo3lzU/Ez1qf2EoJcjNdW69Bw8OrrAFQB9QsTrtYWdtufdjaLCZoBtahMDunh5baYbqvoi6mO6v6QwC91bJLH9768hUizZmKyrKWJpheVXEra1utzW1CY720Dh8tr95sbZnv2+9bdetQcSTZOat5gez625Qk9rAFqTLx1tTS3XgwCKr6S/uMvY6+BerKYSZa/AtOR81oq3wRNo4lVbuMqX8gPeUqp7D5NrmOnDUp9Wc9S0KXcEGaBV0d1d2il2jXvW32bfDMapqhnVhKp1KDjVfSVP9L01A3A8eVpGvFneXkV1rNuxqIq39k7vWQ066HCdQWpVwRIeuX+NCckWBN4lBO3ivxxVuGHo+J4RaNYdnAt0z8XhmvKK5OqLSoGTp+KkWgG/ppGvQDmL+dMuFYZD31uaslPcvNrWU6lR0PwRXsqSr66nm7SWSUiYV6+M2KRhiD6N1ZQxDuFXciemSc7cnEqC24F2QU4y8JxIIS5kLqOHimiS4s5LTt0Cx0d3KR3FGK2zDUSVapQ1Y4ueFsTwFRTJ+CUdUIrwWnG7rlQXEoMWvegnPHU2+pKyPGFuLVTWYMXZgxdBrZVy2iqMEa4avW0CPnpo0xRM4Nw/PdkDYYP3kf1+CVQ48kbpic7ybck5Q9s4wqA6JqPBrQrOw1KbqpCTLMp7znNpzLTRaSPEdDOzKNafWUVuTeJnnm7K/dDe0whDhMoaMw5ya7m9hhkg9KJkYk7fq65cnLcTOnljaO4IFyjQtI0KegoJ47ijfdGyHKM5YM59M4iMyBPzTTLLFptpUbhGnFl2Oiqkn+xhrwtRQgKzYoAmg073LDG0qEfFHm+dGRV+jGip4MZKpuKSrQU7NkaC0jctsoOzziZzMdDSDBF9quq/hbHpYSgl3hc3VChfuzph1ryzjwwf48F+TmJWPOcrLLxZ4b9JaYPe1PkdbwvF7c9TyF+pC8Ms/Xo03j3LksHp5LRhElpBHnJlr2raXpkXnFrWH7XKFB+jiGq5BhvDFrAtf8vvZoMvcDeY5qhyjrzPcSJAaMNwam7tBhYmkWdPcSsHbj1JHRQYAgX13ZXlv9xNu8ueZdXdnEtkLaqLx0aBZ5aIoBMK8phgklB6vjdXXLI11L9cIRp/+aVxbT/TMVIVDo+UgFnlwDUyaJHMOmnIxL5SJp1pNJAgSVnKj7GZwLlbLC8tYSZaUG+70gMVUtmnKtbFbwXrxfS79x4dLtvM6oadCRxrGha9SYq0gUn4k84ozDr/rUVquO8v6icloNoTuiUNehUL/RPACXTXq08jIUgy+VsNSrK99txR6qQNJhBd3dP0dw8QxTfqkSaj/QVem9IubRgT3VeL9Awx26EceEp+M7S3pX7gOGCvH/wqm3E1XtTbAecBjCN8AKPLYiR7sLP9LPQNKOcfofRYHFiyLVV0keqDhnCqcJjFgEVID0UOckHisaORlIL19wqKQzhPoOsoKTME7/pOvYKW9IHwQo3qjK9cfT/H3A1N7Ly68I6aeY8C1nIO+qaOy+uujMi0pUkPYDl/E/VYX/9Ywg139V6yjuQdx/X020YKUTvgf9bW0P0atutsvr9g8314gp5FdVqfwjHRI6NHxYnojLjb/Lv3T4Eqg/pjIhhqNZpKQCTxQcP6RTBe4zjXydTyym7SBpe6ApwJL6MAqu/f+OM7WYdae4fj+ZIYHqmwY1jZmvQT6f+P4g5zZJwK/VQc9DYCr50Yie8HyMqVK/RY4YC8ljYi7DfUCZrmd56VQZgd9lkSpOEe9zrVhvoiyqQORBWVk1/rq64VxmQpUyKr/4nC678+K9yU82WDqJHV8yF8BAPD0DB3/FwGZlKHQW4K76jIWKtNT+fcdFOt/rvAbXaVXk4b9pumIoM63MPBCO6gw/ys+w2sJ7aguZ3idLZ3CBTzPLLGvqyjngzDfRRQKg6X+obvVPH5XCV34ucmf8rRj/E/ofMCX31M5qCVBnYbbW4lo4elCZyYpCOHGMti0M/vwCaqh9nGe2C5e0lsWu63D6soAsVk1tz8zp/IuSo0pppmD5Kj4j4h0qbXg1aXNGocHLig1UddrQLYutlyi24ko8LqvU9b2VcDCOIpfjr4CuMNUFBpvfresrN72ynn4LZojsPPyxDH5CcSd/5tS11s1NPY0dzPIGVZl4VcdYijwnkF9Wlchl4A4Z9HCU7lVMMceFXHgKc1ytVG0ydvDedmWiVqVeyy7KU6cH6B6dM8i3PjoydWXrfnvH2A0yrxPGHSpSXaBCnAN2DfftgyKYbf18A2xrXXU4V29aszN+/6W+5ypO7swiyynJfqI4UeEoZZLUThSF2bwlKdNbVBPUxNWqxf778oiLyImAHbPGRFvxzCyYlsOzJurAyBFyuRifnqavNExqRp+XBp9LO38Z0F2uF9E9R8FjeQQuczE/DmmtMR+NUKA5q+ooZyk7hLpgEUePTllT3Cbkjl9PElVt6KR+rdUEPi7NMnRXpd70TCFVFX/K3eTN4DJ+8jsrKRZVFelOldiX1Um6vl6X11fSuMFu0IGvu3OrEqoUgGkLbuU3AZWkDCmzens1E9OevO7MV2meUfdHevspn9qH2vnjryW0hc0/vtS3n6WLOsMpUKZZOzCVTSMV7OmaqRybnUaJKKVL/18L22lX/7ridh2fT0yeuOKf+Gjmx375MhPdskK7hhytzGo9r1Utx+nPMmw1PT14+bxqjesUAq8o6f+xNfyVu7xm5VxNrjqRW2hyorBesT/1PcBLq/2PK2l2LvAfDWllU6liV5/aPMPP97a24guklVQTOJAlmmVg7ER5dFO9kzSPosxo0g1xJe1uG9WDYTSrb8fW5Cmuey71lK6uudZxbdstI1GvGw+GocyksUTVJZUCr1lXSO5UgQmvqvIJ/C1S3URMXQ6U1wI/5kpgVu6/0B2hH7k7VNlEid4GrBo9VlmFxvasiH2nhlN59UqvrDl3DKP9VkXbctr3jJxvDz6mSzkm8g44165JV/WqupGQrXNI2as+lcbwp9MwaSQFPPB46nOFJcFfIZ3U6mtVSISJhMPxGXbqrGvuHKvmrJztdO67SlmTE9NBNCN9m1PFKL21cHy+QhQQJbAg7p/Hv7TFbX1nANzmJ79Jnz+2VZ79Nv08PpsylbmfvJN1Z9UA18p/p4CcdbmnZK9ytVf+R5d8r1M88KNuky1radZl3+S54ovf6V3CZOK7uzA1kTS5n8WDoOuRslcpZOivXRzhVOl/mxxupRN6RpMA6a8GlH7iTYRZ86kq+fbE51TqviXXV+XOrjEUwg0Kkbby2JqpoFTs1nZ3pkq07Aks25OuaHm6p0XZn1WX8uprRl3nU5BIZRD8hbX2VN7nF6LHbdBNb1n/wzVLU2h0/k/QqKm+lyM0wy7w5NImTH6tA7M2yxpNflbH2lhtd2P6LrTpsw8wcSHa7HDr1EUoRi+0p29DMXyhXTcF0xZrNpHx/kzCXi31b775Orflx2eioMX7rUK+K/fV9ktucurCXrHTk9f9lDdAW3PubOIl+fUl/eq17UJX0yduJmZb8L2oOtjzSFej4Vj7BgsTUVbxzboDyzLwMw9j4YxOJOenlDZicz8M65mYXodu2siNUdeDZbUCj/qJoDgk/9yWU8EU0VEpqM6bUC6JYzeVA32us0n38n9thVJ0z1XqiEOUJ0X8wv+mSZFepi9aTu+Pf1NWpOJVcFbyMYKcp6cnjkolDYaqjrb6SbODVqO8oMPbmVdw+Z2bKIpCTLUldn9WXUgQ9WSUuYs2l4Fhe6K8VrzvMH9NaylOnT7VYZpowhZEsalvC+O0CCKAkj2Fto7EClfkT3kIh+3XcjKvBGi+feyI03/Pv6hDP/27JOfFfBXc/AxDre9nSIzm5rDfuYS5ruF5dNHmeUaT/m0E6EdYPHX1Zs39D9wYbCwBSAAA'}, '30_shiu_lesion_overview.py': {'sha256': '881b0a654be06ec4b6fb295d5a0247591a4b54083f0cd7ffbe3a0358659c6638', 'gzip_base64': 'H4sIAAAAAAACA5U7XY/bRpLv8ysYLw4kY5rW50ijCXfh20w2ARIna2dvcdAKBEW2NDxLpEJS45n4BrDjJL6Dgw2y93LYw11u72WfDhhP7Hhsj+2/IP2jraruJpuUNPYayYjsrq6q7q7vbl66dGnx18XZ8qvFyeLJ4nz5cPFMW7xaPFo8Xbxa3tUWz+Hl5fKr5V34fbH8bvESfgHiEYA/XT7UFi9g1NPFGbadbYQ+XZxA0xmMeQTgomN5D4ieaZ9c37G3thb/Ca/fAPRj6DpZfi8oLE6AGQB7vrwLnJ0ij4D9HvRDIycBJDUY9QLbHkDrE0BwhggEO6/h7e7yfjHwATzeRa61m7/9OMyYNvT8W/OZvbX4CwJBz+ny4fIbbXkPOD6HkYB6+S2f4kuE2KVfQAwMwGwfLp5oy/sA/QIaTon28nt4eImMIdA5DHoAk/qj4ADYOlk8X0Fib126dGkrnM7iJNO8ZDzzkpTJdz89kI/7Xro/CYfylf9Agz3Pwols/Zc0juTz1Mv25XOcbo2SeKrNoA3GiNHaZwpI+sUEVqWZv2ZeFqZZ6Kd5y1H+mCWez3D9trY+3rv50afX3ZsfXnP0UcCCYc3zm35n2K1v7zSGrZa/PWyPap1RgzXqrQZreqP6dr0TDOudnVZjO2DNYaNdG7Y7QbDT1Lc+//QzjoptN2vtdjcYDptt1mwFbW/Uag8D1qp3WH24Uw+a26zhb3sdv4VN3c6QNTojr9Hebg8bo1pX3/rHazf3OC5goFGreU220+o0/B3P227Wu57vDbvQM9xpdIMGsNUcttpdvxG0miO/sbNT22m06myn3anX9a1ff3r9+t6vP88nutMKhl1/u11rN5tDv7k9GnnNRqMJk9xuNBvtoL3d6g67wfYwaNb9ptdidb9ba203un6n3e4MYaLX9353Q2Jj28NOndU72526tzOEFWyP2m3WGvrbnU5ru1Xz6q161x92vFq73miz2sjf2Wl5Qas22hmOgtFQ3/rgxt5vbzr9ds3arlmdmtWtWTs1q16D/+vwf6M22Lq5t/f+TWcCW2okXjRmRrNmmltbWwEbaQkbsYRFPjPM3pYG/1BMHBQOw3VH4YS5rmmDXLIou6o3um66H87dCUvDOErdA9gpe3ak08BwJOXUTvdxNwxEZSfMC9zhUcZSw4SX2QTExxjqf0j+EOkW/Ea6adr77DAIxwwYNN9xCrHiHOG/xAtTpv2TN5mzvSSJE0Nf/Dfo0hnYi1PQNbQrwojg611UZDIIiol5tfgJ1BN6npNheo32TzFmtm4StXTGfKesYTa2uahEfEUmsQ8KEkeGzhfCzRdRt3DOHNE0DuYTVkXFWzkyRGvgH3MX/9qT2AtYYrNDoMbhDP7DESYsmyeRwCu2T6waUeWrte9UdoEPvh1m+1o8YxHBWnoy1E3NS7VRscajONHAPEQa2ILEmHjTYeD1RrSBRr3WaL2Lf0zYM93safv2fBZ4GWxlibl9dSsFj14WT0PfRfvEiR/gNgp2s+nMITlBBt3ImzIuNvh0WbehW1cmAK82TUK/rVuw4HEQRmNHn2ej7sp0kJ4dzKczg+hZIxiQzhPmeqkfhs4H3iRlVhgFINlOw/Imk/g20I94h7k7skeTebpvmLtxao/So8g3oAm2P4oNk7MEHVKggTGx9XzOaeTN0v04M9J4nvjMCmBBwoikRkycd3BN48+oHmk8OQBV3FXgOYiKoICTiscR2GHqBmECmiyx85+rQk5TWxj6XGGjOFPG4uxw8Bpl+5OGLnLxCFTnBHRJOPnX8PIafNsT4Sa1Ch2pUzmHjqPMQwN5U15B7sFCpRsY+Av6ePCrr9CNPiYX+jM6fogawHV/tfxOI6aQyXNt+TUw9oy887lFvFVNgYxn8FFy+QsNbQpBcicNocZTcvAnnMArovWAEDxRQg0OQww+UYIXileKwKknog5NIL+PEQKtGtBD4/QC6NzLcb2kuAwa8nnbhR7I9fXjKGK+lDLbS915EhrmZf1XYCWYk8S6BQ3O5wnIfxZOWTzPHDD+qCnxJLCqaJTdqIL78eyo0C0YbPPZGNhRMgGcl9y9gPX395l/iwVc+TV2CLYuYwF4kWgMvgb8s1AJLj2O4pI2z5iUgjSu0Ia/a/7BsJgOyGcwJLs7B4umf3bj2m8+uQaWMGPjJMyO+AR00x6xzN+PIyDVrw3ecfT4lr5WWP+8EhOX49vXJKkYkf4M8ojh64kUQr6QfpwEqaPydHPvY4hCNDJl2gc3Pv1Em7LMAwvsab//cO/GnnaLHTm/0i1DnyVxFvvxRLdMwTHYNsNUJwsrbwgi4G/rmzSe85srNzD6GGTxJXnbU2JbelXwpndJ8F+RVrwozwc5dcgco4dLJWlYQ/jP3PW9KAjRl6QOQtpjlhl60aiXWDcKkBS2ZerpOAM0JUWH8Mqpn4SzzOVuUC8FFmX4LJ6tAotwNKdNYq+OGnopU+BlzFkGEuJK7OSg5ZCyPCBi86QEXESLm1kZJeyLOWhMyFJ3/0scRVFhGShlLEixj8LBzcjSMGAIpifheD/Ty53BPCHz4E4JFYSZtQpAJrpqdr1KA71NmIYR5Beo3cUWWxibmgiCklm0A5qKtZAwKdBS4ACwMrJK24uOjAr9AyvNElMydgAeEAKXEOIWbDqwASrJUjQ+hl6D6ALDowMMj1S6vRKdNWr0Y1kxwLuAdpxp5GLOyNw/59nhS1Kux+hr0Dpwr0UAr7i6USoJT+A4/oja+Fo4Hp7acl2E3FVDFw2QD5Y/wIj7arpO4bqiTjgjjLNwUutE1apIo0WaQsZwFoN1rLSnB7nYlpeFjFah20iyvD8iElH3hkI2uT+4tzxofMfZbsnt9PmgSNNr9Uaz1d7udHe8oQ9uR6ep+dinxppv2Ko/rZox7s5hNe8Xq89t4XOsldDKPy9yj+dr7T4OBw9+vjjpafrl8uyT+PZaOy81+sgCKYRFs1B7rQkbZa4fz6PMIuUUz7fBwLspGNUoSC0efHMXkSUhBLP6ihsAZJGDKmTuUtzLAqeiUP96R9ePt6SQjCxgwJpYiSWpkBDg+uIEitVFJ2QQdNloQzsCI13cvZHcudxMZbJF8INtqWxba7CyoxkzJiAQIIj5e1K849jae87kPTJSNaUpEU1vME9yqsYIvFYGmUJm5sYC6ytgL0ZhBEGJhKRe8fzeCnZaMTRT6JtsWDYX8zRjZWnfwqb8IOIIsg6PyQD8jPU8UV+iGP0UDQ2KIC/vcatTDjyWP+Rhpuj4CoVatRC4YbYXBLiB5SQUlNnCvVfjPLACxghCQ5ZY2C8mwlt4FsOfweWDraBCA2+4igYE4kAb2vXdwsDk3UWTjXFEnr+ITFOig91Hwv2qPRpQqiFgc1Rl6BWrNriw+vCjSH1OeFny9RVgX7HZp5QbPc4D/IurqvxxpRRBga+cG897Sznvle6VNBzrVsRuT8KIOfpKDpw4MNx+P/SzGwxrC8aopJcJJLRsEqBJSt9x+nriRbfAmidxnLlhAE9T5kUueHyGYQWa+cwDSzMLb0Gg0ay5wr4MepvqM+usJNWdH6E8Lr8tLZ8qeKBW01SUrPIcEt0AdQh3z3dPiRQH5kZOuJ9UMjek/0CUjk4xhH1KO4o++QkmXitsQZoxn2TOncIshqgBaKBYNJ8yXCbBnio5GXPIgBgA2y8vKMa+aMDRZPHuTQtc3jWElHs0kEKsLkM/JInngLinAAW+1Agv18lIEdX3auutGXJXNXT+JE55j0VjrzZrVsImoPETp86u1BuWN0yLtzebsP/bIAMXhT15rrz8hpchuAETYU/Vdy/vl1Mq3Lx+eekGzh2+PD1YGBB7mJ7ew7/HqqnjQ4Wduw35IEPTwpPZfdKqlCyhmDQpbaGwK4WqNyvt7Vxpf59QKW5kFWrqSJLsMEu81KOAzdHDcRQnELbv3raJRQ5l5O/IoEFcygJV5mUpD6ok52K+d/RI7+URV2pyK6D3ilMBGxtkN0okf9TYBLb5OuTHlp4GpQFpFrADdYSC/5d1ZWBJbvRpCHThz8W0pt4hQHmHF0Edy1rkeJywMWpq7sEsMP9iBTClU8sPlK+SWzNtSvdEVdbcHSfxfJaqxmA1RnLz+Ejj4DYEWcCEB+JkoOM3rf7AtL0ZyElgGJUoQLU5/UFOxsDIUASFpkV4KbKKE0iSDEGHrJBhWhiO8TqudtgzDnnCbdVkvE0N9QFfpVUrYmNx9NAQUCbYDkWvMRR1uBD1DxEJMneIrBAPYNooQi1AGmtAYKpZAkZDsI3JAZ8fKAWuTU7NjxMwN7wcittZLWYQAvOXTrNseLLkqFcaq4ik0rx+BtZ6pksE2KHPZuqBmX0zfyRT1wNTkKbKPKazCQOXUDDtOM0apDMBT335glCjMoga3XzJVS0scy7H47pIWoUSqF4pjB3Exz2SPriqEimNRtZKnWvQxROwYwJIl9LFZZSD68DxdA6rDYn8OIkIRvQD16ha9o2PfvPh5wJ6Gu2sA8FzY67YkKNzyDjbZ4muTCy65WyUZKE1l+sK7jWzIZ2TanknL64cocvucfHkg/WeQALLn08fw6cJ+hH4a5F7ccGv4VB8tlYSUWnr0JIfYLFRQIPV6XP0gz53TQOFbdwVgNhkOVXEIoboFSJn6XlVRXaCP9fllqugKIibUVM+SiENcqzKk8X70kDpAZ8g28Gmq0PCYgTYcbUHTPtgM32eBOcM0GvBAe8VLIg+zoMYF5aHhcooyYbsu5gP4pwPnDEvwQylp9gXSwqHq+6F1FUKPMKYlIvCjzBeJ59voD5hbrcG1BOfRYDG0I9A8ClaRITvOXZXaEwU62ZFkARNnm8jLQE6B62BoJAFemVAyTjgoFUDoYdRLk/V6ej6sbkxwppNYgqGUyueZxaW8YXXKa42IAheh6Dacv5qzyFE1a+Nx/LQqQpvz47wCaOtGZAT7oMOCiCAAmtxEAZzbwI6qPeqxjd1+gmZ2UQ6dWnmEjDAg61KJUuO6skpliwM2pDUES4bCy8JuNgCuxxsmrsI6WBoQ0PK/jydT2mg4xyuG1vk8PMh0JDk1GlIYBITwoRkBmUy6ASVkmE4trxDB9bPBry4nKkBbWn4JXOMrtW0t4Fr79BOfS/D0JV86BW73ih8FOeH3GtzsNqcOvWuNfGGbOLolBKfUoUCbL++ivnyJsytN2EWVZICdz5FpAF7ciggsfCmGVRZecrrKuKWEF4CUusmy4embh1J/H8luBf8xsFrOo58zuu6WLmpQ6tuZWEGjnO0cl5kaXdwI461xX8sv7W0yLlDJW+ahnnMl2HCxuiezF3MCnLn6w6PXGTYnkVj7hVR4Qpp9g68EDgEquvEOelz34VHW7k3JwPRV23UQDUVK6Kfk+htEn0S6JL0l32sogo5MrMUzI6E8FfGgfyuHVpVBXXuOZhYgBWMqBH9XqM9eDsdwLl1LGW/3rVbXdNqcbUYeom4gqNAQKhbWeB365AJ5yxK+fXjSZw4+i862x3WaXAx8A73MYs0urW8e8drd1td3cL2NDvC0OzKFV0Kfrf2D9Va1Pcrwu8eQoB5K13LKtYRkr4a2AzMFV65pipatPhfOuL4iZconxHll3gUeqok/4r6/Jeq+Ja2hudCe35Ub+XdozrP2eKspERcgbHaRlV9eGy0zbWKJM4OD1JXaEChSrDHdkZOfuIdgXOCMdiUegfgJccGtFxFHFYwC516q2buooTwygn0Q0LE9QFhZDYaeZOjL5m8JIIOD2uRomJKXp/ck7y/IQ9W8Bib6qnwYO7CH3t6Cy9+8HtaKT/vpusUbnyL3kz1zkn1cgpyrvNnV/blV0XM3TxTdkqn+ReP4pd74pmTV4aLmfHqMEUFeVslEBf59aZ0nUcP3m2nvxKfK8F5JS5Hw6j3UhFoUh0LIjEZ8Yn3xNLV4xSAZ356vJrhi/RemDGqrAzeMqQgvlXz690WxkdwDmYnZ3sgSw3qCN5UMtnqEF6HyitVtFE88qQCu1WxcvmKiSUqLVB5eSqLM7CAd1MoB5sEwOVG1MRmKSGyqjmPJXOV1QRFyU6sctpRyjTK6UU5pbAq6UI5QagkBZU8wFoX4q+E9aVI3loXl6/dGlzLMKN7U7Q9fCl5bUecoynJRCqrQYWgSesgs8LylT7ckPysrVwfYZNwHFajgZJwrXOHeUggBY8n4W8fKRRnwqXYQKnnSMZWSzp5WYcj2FTQAT4qwqVEFRL7YMXxroExV8j/vUWf6gZeXFqQhN1yOp6viKUL+XPzCYKzoknoPVqU44LliqSpI8RB5HqDsMrERWQxNCmmJ+0BEABzsMlBWCWLVNWBwWWjv4nbAS9PFQS565DlQhSOIiademPm5CnlRuPMUXDJL/JONbQlTD0+LbmB1MahhEx8RDmnFIIEz3X0IgEVhxv36UDjhN9XpHDk5W7pVGz5kJ+APKUDxJf4zcUj+nph8RNd7vga4pvneMXKoiuqioNPGNJ37ugonHPwXnhFzdJxNnoP/1pyQ/DmGBbVsEyfyAuo0Cv2SR6C9sSp6cXOvlK2F3Do3JN5xCkUnn6j4895y2+w9cjrl5HzYoHeu5OTUWoq5IctrD1M5oFS3qI+cFWqoCllrtQs/IurQmCaMe4XvocbiHFhJs3jCnuqYJaKQBB3kPCAnuNPdRgEkV4GnX198f8YEb/tpVCNh9Hn/KMTjJBL38SQYOHp6/3lv9MdIPV2EB/2M7/QcrZ4TB/nkMxBErr8t8UTkLCNVafF/9CJ7iuEk1df+Q0DCtF/4heGT/GGwvL7tV/wYAou7iRQ+L78RmvWSim1nAX0VON/IiZPzM/k50DK/V76/uYe4flq+fDCmfwZGcF1oszhmVjS9ajEdUj+vQ+QzD8yIgx0G5Im/Qw5ps2jbxGEytJHRU/EhQtcITxsPi8TOFmcv3HdH9GXWnReitmNPDSVl4qfcUHZVa8pyU+aznLWi6+TXomD7xMSFnlt5BzX9Uxcn35M56p0kZkuROdX0PAU/Yyfw+IUlt9dzDy/QopfQ4nb2s80qrCspZJLJpdsolV8RgbDxYVu+fGWUmehJnHX5RwPii9k60e1jLNmgV7TRp0ifyvLfM6v85U+HAMMtHrqlydcgin/fU6icr44sTSUNJr6z4DhlASgWHt+VZD7CxApWunTopdfGTwl1b3Ptbbg6GtSHsR3RusCjbY+4OeV6hcZ3LLPp1MvOeKXayzuRSolWmwSaevUC6P8cyFHfrdmX0vG8ylEuZ/hW4KXyelOLZ67uW4Q+64LKTHeJnI9AWnoV65wK45Ev5iHCQu4N1sDCC7jCroTSxyh4sn5GjDydZa/H4c+BSAyOLYqLn9Q4Mmz611AlDozm+aDWFNxYa5IuFObRQdhEkd9/YOP/9m98bvr7vsf3cCrHG+fhpdCFOGyZSkAidpKPYDehSfFfJYcJTWWQxQebIwulT+UqNwRtcrWFZ562h3OQF861kG/4lgHx7aGRof7DBiwKjHHl9RIRBY6ampktEc/+NWHl2KbUpO8WBiVWIZhXJWHM/kiWKKjN9Lv0C1AQG/aLn1M5LrHwDE0HOu5/+0Pjqvrln/UaIN/h4DNhRGGaeEAJz3i9xeSZN0k61tbEM5IYhBYui6qh+vqPfxqEj9qyQyuMObW3wBe/pYEejsAAA=='}, 'fly_colab.py': {'sha256': 'e169c93c9b3162bd5303172038e5129956b0e9c4ef5320d1d03ef200aba399a9', 'gzip_base64': 'H4sIAAAAAAACA8U8a2/k1nXf9SvoDQySuxQ1M3qsxDFdrHdle9F9ZSUbKaYDgjO8IzGaIcckRw8LArzrJNvUblyn/VAETdLkQ78VWK+tWvv+C6N/1HPOvZe85FDS2kZQB9EOL+/zvF+Xly5d2tgOJ9ruymJDiyPtejz0e44WRgEbM/gTZdr1ex9pKUtCllpaOBpNMr83ZNqNJNxlWn+b9XfGcRhl8PID6DgOh3Fmz81tbjMt204Y0xI2YAmL+kxL+0k4zlLNT2BgPA5ZoE2iURyEA/zZY2kY4KAw1QbhkNnanVhj+xlLIn84t/HzW2HGNHgXjsZxkrHAntkCDePTR2xP2Su1W1oUZ5qvDXHQjffsuUuXLs3xyWDI1thPUiaf+3HUnySw68weTLJJwtL8TborfwZ+xrJwlA/a+jQcy9/bfro9DHvykf8DDfYkC4ey9ZdpHMnfIz/bzn9Phlk4TuI+S9Mw2pLNcTo3SOKRNoauMJWYVLunjEy31fnTT4YAs8X88SA/RcZGYwRK/qwcYzIJg7m5QZxo3g47AErQDP3u7XvenY9ue5sf3l+/dmNDt/S799bvvHfr2kal+fbf36q0wNP6L+7dL7WazpwG/8WpzaLdMImjDi7VdfWmPrd5d/PaLXf1cqvRvLzYmLt+986d9eubN+/ecfVWo7XoNRY9+AsIilg/C3fD7MAD0vUGIVCJDUj8ZMIyfe7O+kf3797ZqIwZjYcsYxGAVRkDGNXnNq7fv3lv09v48Jp7qLeWvRR4wkOewJWyJIapD3RH7y0PWo2Gv8jWlq62+mu+v7LYXPX7fm8V3vTWWqtBa9EfLPaWllf7rWBpcdBvra011lpLTba2fLXZ1C06uPxPb13lK2XxGOalBflCDH4tL68Gvd7iMltcCpb9wdJyL2BLzaus2VtrBosrrNVf8a/2l7Bp9WqPta4O/NbyynKvNWisziy0yhcCPgjjKFVWGgQs6DX8/mL/am+1ubLW6i0t9VfgOI2rgxZrNZdaDI7UXGleDXrNq2tLrZWALfZay43e8tUgWFusrgQgU1by4l2W7IZsjy+2utqEtVaWl3qsscL6S72VQa+1tgzHa7SWri6vNf2l3vJSY3Vx0OgHVweDHqzdWFxeXVle66+sLK7qR3PejWub19w7ccTmvPeubayLn7fWN5BG6GFuLmAD4AXfQFYRxLbtCp604QXAyTCpeS/MtomFeF87Bqln6ElPNzUf5JCTHw4ZojeM+zvIEcBWiTH0R73AdwZ2wvzAaMIJLuMf0+rpQOHatj0Zo4gwaBRfLWEgTCJ4tc32g3CLpRlsg293GPuBl8tKQ+x6EA8Dlri0Qc9DlvU8ExZM4+EudEKCByE1JzcY+SNmsf0xsAZIVNhoQdc27HmUynnxv5DDiC+xgEPNt1w52NESP0yZ9rE/nLD1JIkTQ5/+cXqsnT6Yvpo+mb6ePp5+B/8/Pn2oTZ9B08vTh6efwb/PT7+cvpw+Pv0Kep5+Pn0B/3vMe0B3R9Ov0EK0iRSWcsui0cY2D8UcPyyAzs+Ajgx9MDwAXgTlJChLt8TGz6RtvghvrS4DWmcC09NCuKSBf8w2/rUREyyx2T7shPcz+CR8QuBVlz9TT+Rdw2wLzPIXFrTh/+2enzI5h0Q0MEQ4OKBhSm/saQG5+BaSKWgrfjqBLlBVgIgQyQkUJNPiJNxC4YWbAR0KHYEkUyCvfpwAP2tCrI6ANiwi8XgCqmvohyPQJ9AVkAtUCKRmoxIkYuDCP5qMxgdI+dFYECw2u6ip6LSpIelFTyejkZ8c2PhON4kJvAz0tQH0Czo92nL1STZY1U0ONqA2Ppm9xTJDTzM/m6Q6UJwe7+hwoNJbABmjd2kIypAoYKYPMAHoSd06PDJ5Q5aE/pCmXGwUVF5HxkCX/4tEqhEZvwbqpJ9AvND6avrt9JV2+hB+vJ4HqaxRT7COqMcTfEGE/sQWFAbwcNWNgebOYiBV2hr1AJQTabqHOw70pm47JnHsDlew4wPAEdC03oNDRC34QZjQzSMJPD86MNCACcEkAOChjNi10iwxES74Ypfm28X55Hr2Lh4bmN48i52Re5/AP58BW59Mn2rTE+DaXxOzliExfSyPKyUEKMoUbK+RrztNS5cKGUQ+l6+6wwm5o7xK9W5HF6+7cEQ2SWr68+Zy35KOAS3DTcl8qODHzbv3UNQBFJHv5EtkRNQU+AoarMuXJXwsHU1O0EvATttASvogYWA/RP0Db/tT3QHkzywcTBIiR2+Uwrkb0EMPMnpo2ACGlLEgzfezsb5+YyPHIGD+Avn6ZyQsgjfKy+caPXxD9PYYyVBIX2xFpD0EXMEvoGbC42NJnNDpGWDzhUYS+iWI5i/w4QTI+0SI4tPPJTr78QSMdxeBRCxcmNOC03N9Yg1BNaKI6ughHLJbMHZKBrnBpwL+U4/vnMOBp18A9cFuj+vYa4bwsjjzh240tj9lSZwa1d1YQXYwZvge9r6yZOZKsW95yBR8dzlLOHy+K26fM3GCelYcBOYYgNSJ4giXMqjjuw3TzuJhiCrbAmvV5epfCx1jHoFFnToh7AOflI1hWwEq3DWtZb6DsiWXaR0pz7pciPho3DJP8gIBVY48e1R/ksWDgQckypCCYVSxNRrbaa6tdbvmwgUS8j8I9t9WpOD55Kei7/H0hcQaGVi50uCGLtncwtYqaYv51fk03NItcN+GYcRcfcYMQxoFRMEE9o2wn92nR2NgqlYN72ODVzkM0NpI33IBUH60AxyexHHmhQH8GjE/yiFl6QQksAXCHZZ6YMQKbdI9S3JOX9M5nyBbFSbPQ2j6Bvns9DfAssVpJTToCPFe6hIh8Y2WaANfAtpgWO3Cv5dccYzMjQoLFdkrbfrN6ReAkC81zjvUhlhDFFHTM7kD5AiEhYVLWaGJrMFA2TAEhQEuLG3B4tSCIhAoulm2GqGDgGf3LRdUkIG/OVXSGwFieFlmg7zHWbDms5W4KVdv6CDbYdofxikzBmCLZAbNVUKj4L58NNC5lbAhGFtDt8nmmy3L76XFk3KsMxjhryV8/gR2cBRUwEz6lRxuJc+gU4EYYitEFCn46FqoSoQ1OQZBAD6AgUC30FoeglEQJ0wc7QyzTjRzYwPbey3FWM4NUrfqlPDB0XgCGgNXXNAbTQ+kRW4zgKY59k5/BSd+Cmzxhd4WDgwfI/lfb9NOPWp16feCzruUXtmjnSBMDKmoMGLkHh7lZIweAvI3mVBFoMASzr+C3TSeJH0mdyHHqTSNJMZ7AZGR72GcZzTJI3K9DH/+HS0nQO2xI3DLJzMLtg/A2wObHY0HVzljvpv2OEHa1ad/IAY/AT0OpDR9AY6O6GHpfK0nQEbwAv7PdftTlAW0PA/f2bYNg4aTdNvdTCas2AIPD9kAyQM6Id+jpezMbHMwd+SaXVd5y2mE9k5ekHKKCmbRnFCRJ51TRFWn7DqA7ZibHLJFURPdK52BznW3h9aVdxg6jVZwZEdgHhf8Aay0xYzFhtl1qmgtebgXYPf3hUtwLMQ3Z3WEPCD8OwR6jv2yO1sHYWVpK4fMQjGG27xgSqtGsoO8B+oLTpRwohBIKWgcBF1uJ5/ZXfBBt+xECDN7B8zrrTDzesO4h6ZyE8Uv9yF2LPIiaNr1X9yDFddvyPBBPdj+tWJ6HmuFDOCh5UJe5rGD098hIz2oCR+gvQqmahFAwEepwCSb/FmoOnRevkJ+KGlfNC1PpHWpCmN4UYjxWk5BEezS4XeFz81pPK1Bg1UBtRC+Oc3vhFFg5YRvlNBsFfOYlpGjMxdfCiWjt+yKpYgrbWwxSlaPgU3Q7qXhp8ySDyMM7XoR2hQ55mFPgPzeQcZAy1jlVtkfVKlTbxmqsv5pjmd0L74XMgktogdgDQFyXuXSCp9fFyhDMimzT9+PAoI2Gh88vEFS5g3jJTl38dlGDJD4/+ShovjyfoqHqovgaWWOIpjXOTPoNbuRQrrmE2HYr5BGM/LXtPjAdPesEaoZX1pQxDGc9AA8Le5l2+l4GAKtdhrdPLzh9Fq254kOniejHQ44XqXmgiZ0R6EPdUXpsodgSqLTLgD9/v31n2/U+uMzTv+5Lv2Rap3V0h9SWolihWWmwJ2M/WpA7yOQKT14s80CzcZcG4bmigxWuBWBJAnaII5G/hb0AY25A5BBe7gH2g6mT0EuAOXIEB6P/qWqhYQxbZQ8fG17C2S9oaDbuywQXkiaLDkoG8VhNIjV+B9OeWGwTxFLOJ47p7AhMOIprDcYHsxTMHe+P57MFxuab1KYrxgDbgT0pzVTUEA1ImlmwUqQrJhL0LJZiprV9kQPSCdXwrxwRQH1Do5Gw13vdl38nXdi+302BiukmMG6u8H/3TwY8xbycqHjhf7If6L1SR4EKDaUtC+K4CUXtP/LbRQeayfAoYi9otua1NKoUOHfk9NHMBnX1095qOkZvtBAjj8mUX18+tvTr6cvQfFqlHiEDc4pQBYndyR3iGeLcjCkqSk5kbqdXREYpWfdLAKVYkgelOnK+cVQmww/A2cEMmgi0tCEGWMeGLeAL6Al0MYcmfmMnPqxQcxkXhDveCBctRNKXOSgxTgUmi55dPhxHkc7IW1Hvt5LAUhuvjxDrfeMZsORT8+B/CsyTkgrSm+eK9BjRMgDMmheksp8Ti9e5VG77XAYwMEwotzpFpFkcfxCAoSYwrdKwK5LBO12JHa6KmydfKGO0qNr+2MUWAZNbqrGJYYwds13mwWK5QQXxKL/BIYbxlA+o8jgMY8Nfoe2Ag9QI3GeVDADgMWMlKDXzwmiYIxA32M5Fs0/gCMGt0/IICRb/tX0m9N/hkXAa4KFvpJDTk5/nUdRZ5nEzhNK/i6Y7J0d1VbOTymAK3lkt6sGd/hQpGTnh1Ph+fT2SljRj4higKPldkEDRm4K7Aeu3SRx+RZAF/PgHGybadBeQrpKGPQuollq9/zfsNYzNP+e8NyB2P8srpQAGE5m+0FgwOR8V1KIwu9uQWcq6HCMCIKKzuaPgGGNmJQbqwixAk5cm4/8HbDXIn+cbseZEfREfs7iey1la4XjK6UfOhYW2wcTwIt3FDfDs4ZozIBdX28o5KenXqgEcb7aQ/8lP+VJNRxRscsJR18Str4jXvsWIPK0xCbAVf8FtP/g9NHp10hsNXLIkjz55CyWQt8OfQAJXZIUrmp66Fdk/Yyd/4jiPSN/wD+fAjnak6xvgvZPBthi6G//w9ujt4PNtz98+/bbG7p5RYepsGbFxj9Lhom5dZ4v8BNQO65wwbmwAj3Iq2LsrU91s81tqpku3ChqC6Ns5rVoVwPcsqDG3mQYWvOTgxshkBI4gwcGKXfsoPhzgpB4Yh/fmQu6bBQ7XNTz7rSEaLWFs2LI7jS9EqRxtKBn9/z+zmRsqJGdfDbSqS50wgz3JAOQbqzfAj9Uu373ozubxmVTe//+3duaTKfaA5b1wahnZMJX9iS3XC6bEHElgQHxcl+8TPy9splDM2HxlP0B/HmfwibwJ+790oW+FuaCXX2vp1vkm7oNmgV6jjGLVgm4wCAZ1eI9rKIuo2Jb7dnk+INcjFN7kB5EfYMaMfAVG6bUs0Cw6WTkogPEjyODoGAWH0qb1jnPorXQKnS4IpayzeH/WtzUdAgllrRQHblqydHpg9GdscADdtCdH8Q7YRrzfRomusEg35Am4qTk3ZVLTETIiFAjyF2iUbeqNj9P0VA5mx2AI0eWtDWAfukkYZ6f9sPQfR+IiVmYYMNaLy4WzfZgFgmDCgpABKruh9jMOR7IW2R/o62oIO0tVwJVSlBhhMuyls8pWfqcxN2XqCmOyQJ+BBLzIZoCmFuR2gSUbVXO5sUAKWwNLW65U4vLGFPTfqaRYCEtBEYHyFKMW3E5yWNSuTbHHg9Q1qplCU+FpUNLgtx9Ic2iGjDxRS+G0hsCQ4nbnBAoMIAGvSjYxpWryDR/fvq7fGdFkpnH7AaFpsoh+EEcb4EJosbPD4kbjhYOqRTwSDMOiXeOzNlQnVDa9F4o6oRRCK7Q1UJRc/7j6l2NeFvCqUc3XqjxixWH6gYJIZfHlWcIj69ZhIrqU/8S41R5ccJ18qvTX1P26CW6YTLgViLCkkGDvdrnWLAVN4/09hOYVJBYQY1qZYsCqgXKNo+Y3sYiFjW8X4hwEhL86FZVHxgwbEHEttJcy5kV/TA4T6gPqvJclOmWt4lxGQzQ6G1ZxatuNujJAgVc2At6huglQjnDIMfvj1SSBc5JvCuBXJiQZzDNdg0R/IEQITBG5tWr2bIIrIwozNrjusg64rFi1wY9tc5QVmVKL5AHGi8qM1w4r7jzx1X0yRl0i2/BbOPTT6zWwylKh8cGcXy2jzN73KhWsqYCENWEJ0rtan4ir2ZUKzRFNrTlXb/3kZdzFDAsPjxHHtbVQGuqqyE78ATexA9IMVwl/PJaL2BGXoBDW3GBsJiftvdMlBSplIKbyX0xXK/b9hNwbXdZLgyxtV4WoswTvX+S0DtPuuWlQKOxTBtXxUmdPBLbKgSSb8EMFbnTq5U7vtWrihyi0Aob5TkHi6o7iCmKcioWGLBeXZ1HRU6cVWvCVe9LiheRSYJQu1g2SHAV28Esyhsl7vOELleMczKDLQl92Zv+D2j4B2SqfHb60INJwJ7xyFZAR/OVjGWdfqUvGLrgu1lXreM0W12zjXPXOM6qi5VjrMAiOTgXq5Q3QW27N2uK9mZNUaTyMxZ9izsJgOgfa1Wd/osK0rNMzNyYUjKgvBaAapDAKHuMhiLYTTmh5cYUbO0Qt39UtqW4bPR3/XCIF2Y8yg0a5RqScUp3S7CFOrq8wd4Nk2ziD70RG5G3a+fT5EHIYTgKM2uS8pr0jqEvpAfpwiBd6G8l8WS8wIfaIx+cizPeCRWtY6a0tofsSGt5YcTPcNZ8svck9bdY0VvNuc5kQoZcP9ICar0zBSdCLAKfULkd9cLD1naayVcM33J1PLkjwDoKQWDhL/Ac9o0GFTQNzfmJMlKmFGQeoZAXgN+xn6aq8qO5BIb3YnQLDHDAZMRKFrxawmbDhJkL79UZxOUKm4wdg25dWHQNwyqG4zhLOsjiAoa/y/gdG4yXkaqlrJliaZeV56I3/TdR//qEy6NSQl8WEwkxURNSiyeF1S4qI9O/baCp46yC8IorpUp8aVfCjQAhtoOggO4FJDpqkrPLxxsxVtRUCuz3EhAznJhyTzs1DmUlvaPL+02ymFF3iOctWSPq8H+tokDdwQ0c1fnpIV39c1vmjLNfqsEooatSf+HodNBZMdMfTzx+n1CtWgNi9zh5phYDvXpALdy+dhuCXjCB6Q81lQQFkHlIlf+mBK3ahxQzprRZtOvWVMwVFCQP9vu8Jv5EKReBI4lZOjLT3S1aeA5baRDJ7q6lt9Uq+xNxyVIX+M9n4o9yGv6Uz1F1fX+s3dn+QUHqH2qglutEVUtSFCjPRp+E3qzEn+qrsyQMMWDyeV6B+5zizb8FHfitSL2houW+LnenHudxFmysca0xvGJrQqP+sFSccCnPijvg4a0SJATRSWIt7CsGnFcAMC5KIwt/VpE0bVhUShjpyCLDEbFTxjSfirB3Vgaj6BrAL0oV1bi9tYqi7AObbSXMk8OXAwvMDqrXx7VyY2Q2niOMY+rmutRLCZrP6hPO5ATBtupak95STJUZy6YtRI3bwjUV2eNCCyaUDTACUVDxBDQFdZrmu+IlTfeu22yQ6Xj58iKhTmuWzDPuadGNjt+I7ApQDwBCrHTURjp8gpkTcd+GrDd04V8AQT+UJiLl/bj1rx3SygtiVcduDo40EMIn069tun2NLsAr8gBEEbjM7fCUHA/5nHATk6KN3/NyZR70mUHHeOhHrmEMLEA3z9pjPUduD1CNDTVnvK5T716ZVWnUIVXHURkOlayJmUXuPsgpUeCEU5YiU8Rt6owlWRwP0/wFbTRvtsN0GIL3jq2KFhFnEkmdw6N2xPa8NAQ/3220xzDQzSsXSrbfz/AEO3lImEoXKTNW3K7h9aHarTCa7KMBfjz9vuyc5ZepHmGqmbtFbS6hMA8pqx5FwvqhKF08/VoTIu4lXcSi5b/BGCssf//abbsAAG5/9ta6fY/fIr8Hr9eJo0GOqtQuNe5oTBedwbhwK7fPUW7Ld1TDs6OXLNFtf5LibTCyG9pYkDRG8NJjkSyiPLMschKlN/lYp66ER85UMJzKv++0JOs5lRtahQ7/I+FKyG7CwyMOxzLXUWA95zqsSqQwO9fX/4SqAy9PIXc+ojzAF7MFpTzbXagUUY6qxDLKzFWACR/bBRDxsZwbI8CVoEUAmYEQik0BX/MdgdVZ0CBda3h5KUJsIn+YM32Ec7EBs9/MWCJyiuU9tnvg2ezMQp5voIPUaKeTHjhKBt8L3pgyu7ljUcG1GOdos9NKo9by6sh7z4cVZKKFS31vb5tFdX3fv3l/Y9O7fvf2vVvrm+s3ygcn2UYd+VUxvugsAFENllwlmY8ax2ODT2C2h2wAKh+rC6EP7CRIXf7K5ha4MQt0VePevLOxfn9Tu3ln867QrdrH1259tL5h/J2l/M/ULaO6HWt2aWln2Em852EB65sMMk2uT+MRorBmu+guYQXHzGQwMBerV9zmGcw50GvMAVCNOBtqtNPfWNohnxWlxaVqCPbSkaWRAXKIf4/qbzuoRCa39K5LvoVzvkmEL01VPczVzfVTJpmxci6wbuYUznwPLMF1+omcWb+1uRmuv3ivczVSIF9HFiiq0rUa0VJloMwTvq6EeGdlpxrkAp8IFqnFJrkD3PNBIA0PSidHiePMcLTQYsjS5LAYQE9+loHbA90t3RPvMUxEVjDSGl4nL0rV6ohJjMKMn48fk8GrJLINBOYI3RtWYRouEbcnWRDvRQaKLe5y9bHydOgJGVU5cmHRyste+HEd1XEN2C4YOjJ3cWZ18hu5viLrSZphn8C3j4CrMegABvvSZqN7EvdvfvDhJqmhcvvtO2sFLNXvCRzqWKAColN3+BGsIoyRTKIIq2uqcQpLV4usPeUTB1RAjw1vXN1gyY8FOJ3K3XJ9z09oeQfTExTIznOlZxjOmqws42n6F5hjmZfe9/S4zW2474WBJw6ukV3wLdhyD0Vpn7y+WXRFxoEOjygo/zI3PHJ+oqw72ZP8Bio3dU6EK1Cz26rBicl6sikfUx4fzB5wVU4wg/+Eu9FoulJFlzCgoK+l5XfRfwWzvpw+E2MIolRuVrjgrVWbZ5yUSyA87YxFg2DGPpKfEHnNd08CXvrd0ha2hG9TzDEz/Bnlu1/T1ZOv+IZs/aj8XZQkDIpbOPhNHyAxiifLD4qI8KUSA87vGklK0AXL0Z2eWjmFWWPq4ro6OJA6SF4rye0AkXjmkdSCS5uNhtXADSqWNQYDSmO3wB09cyBn54XybTR5SVzer5V1u3TdLAjpsxoOv1nIz647OFXlMwi4Bl2k0J0GXk0ZZNwr1h2QoWQ2yOfEAvYZDj2xY2A9/uNIZf6OZPWuW4QsZfgW74OKkNaSN/1T1T3yPqAoF1HJNxgR5kPeMJiF1fAuH7FgcCRRUPdvHB/OCwqVkiqsy3/zeioOuPqKqjxS+ya1VDP0PBMKAw2Mm6sJ29aRX2383+pNwmFQTh7N3DQutfcngY9+dhR/4jva+0uNpgXEsgUIxAsv+NbDOxqBP5QO+s+0db+/rQHJD8lap2szKWo3wNrwoM1VpRZHwwP6jhyokhGGE2MMLGT9bXSbUDAlMGs8AlcKHI9Ryv3pXsvmxAHmehih/duGphTcYN6Mn1ko7QiEgiz2pAvs/PwcDF4ceaDRFCrEuQBQmZf2gQL45ABfHxyDPn4pyg4yt2E3L0PziEc4xqAvd72G7swvt6j5YwuekzSrtGTb0LC0LBsyb9RL8CsmfCrkcRBHy/mTlwz68NrmM4g7A+wTVw92F4JMczUDFtXmtV3tirZlagsaTag52m48zDRjEg3RtkrYIPHp7OY/RsEWHzm/hd39ybmdYXl4z0UEFwCRCzu5Q5frPsBk3ew3PsDZAFtgOw5cHT8Q4ScgtbfBctrGUiV9V3tXIzCALwj4wgZXI0i1tS342bg8+hjfyU24OoKAZHk69kFkixx9ZIOt1CGYd9uRveU24C90xUYONl4HCzyGO944QIsajJzIikACikPrFqAeLC5X39KuuNoeqo6hf+A27VUOcLMN4/Ny3lB8LAFG6F3rl/IpBiyLrNAeQ1mbijfiCYQ65rSNPG7GedCRvTvKrK4YOozjnclY73Z4325X+Ca4nT1XDLzcsFtXBS1xKozD1O10i49IcHVaGICKQR7sV5eC3gDKDrzpEiQbCn3L2aV6gjf34BHE3028d2vwYUDhYKa7TQtZ3M0lD87z4acW37XLN91a5tN/LMTeKOZ4wg9O3I6BpeMEcCXqXUiMoqOWcerLMGIBrwEaFgy0LuPWhAhggUFO7py4lAumbK4iDJrDBmY38BKhYLOCtg6PyjWLIIhhen7DqVOBlrSeO6Ru9W5XfNjiDQbwW400AlgIoKlscJ62bC2a+aWGMJJZfvr+ZeLKL2Ha15KtCRrX96gdDsc7oNfv+eKdoc/Po7ZGnvpkAjIwEPGtM/rSZ72s/nYMUhRoieel8AMAYCfR33kS2/B7S/nNS0eAKcqLKLueWUgEwEA44CeBAHLFok2r1bWEvEWtecYMSgU3RQyUqfLBy2cf1N+fF0bX7LgG3zr0T10xnv7BGVKjVFvafAcRktpK5o12847b5B8PordFiPudxkUlRPxzEXhH4yE1YN3u85q7RnTF7bWIdx/jBZ8zvn6GF79oE4Bb/i2zN8GqORPUr36i5DzjofQZMkNv2U1QmDA/6DJ7RTdLewIzvFiW52lwxJrdgAFNu7ViL+nle/SVu8nlO8nnfyJxID4NRJ/Yosoq8EZ+y8MeagL4PTrRwp3J6N6Box2WVzxaOCyviZmiUuyfvM5DuY+jvIAvljdK6PT4WKImbMCoBaUPz/siXHF58UkRHn9dXC+qKdoWH7TBJd/wWgyopEG4r94FJjvK1Rco2wD+BO6bjlM0mfn+OSp52Gbmfg05Q8rlGhW/KmUI0eKcXRaq+GOVsZzMKyVCPyYEwxXQXF2UtFTiRRWbamqC31/Tpn8tvkr0ulwShiltHocAKhPl9DOMrE3/grwPzrzwnfNMYflipbg8dsa3ZWbhQ85vfbEH9ZO5p1oJZ1Uk2/kLCfZ2ZiNktI2qVz3ba4t6zcHMnocq2/NgYs9DBel5MC/XlHP/B1moAOkuWwAA'}}
CODE = Path('/content/fly-shiu-code')
CODE.mkdir(exist_ok=True)
for name, item in BUNDLE.items():
    contents = gzip.decompress(base64.b64decode(item['gzip_base64']))
    if hashlib.sha256(contents).hexdigest() != item['sha256']:
        raise RuntimeError('Ошибка контрольной суммы встроенного файла: ' + name)
    target = CODE / name
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() != item['sha256']:
        raise RuntimeError('Файл кода был изменён и не будет перезаписан: ' + str(target))
    if not target.exists():
        target.write_bytes(contents)
print('Встроенный код проверен.')

In [ ]:
#@title 4. Проверить входные файлы и топ-200 без симуляций
subprocess.run([str(CPU_PY), str(CODE / 'fly_colab.py'), '--root', str(ROOT), '--mode', 'check'], check=True)

## 5. Основная серия CPU

Включите флажок **START_CPU**, только когда готовы к расчёту, и запустите следующую ячейку. До двух процессов используются, только если хватает свободной памяти; иначе один. Скорость зависит от выданной Colab машины, сеть строится заново для каждого повтора, ускорение относительно вашего компьютера не гарантируется. Для ориентира: прежний локальный замер около 28 секунд на повтор даёт примерно 380 часов последовательной симуляции для 48 240 повторов, без построения сети; Colab может быть быстрее или медленнее.

База работает на временном диске Colab. Каждые 25 завершённых повторов создаётся новый сжатый согласованный снимок на Google Диске и проверяется его чтение. При внезапном отключении последние повторы после снимка могут потеряться. Для продолжения подключитесь заново и выполните ячейки 1, 3, 3б, 4 и 5; в 5 снова включите флажок. Уже сохранённые повторы пропускаются. Снимки не удаляются автоматически, следите за свободным местом на Диске. При ошибке записи процесс прекращает работу. Сеть/Google Drive могут задерживать синхронизацию: после остановки убедитесь, что снимки действительно видны в Google Диске.

**Не запускайте две копии блокнота одновременно** с одной и той же папкой: блокировки Google Диска не дают надёжной гарантии между машинами. Если возникнут две ветки снимков, блокнот остановится. Не удаляйте и не объединяйте их самостоятельно.

In [ ]:
#@title 5. Запустить или продолжить CPU-серию
START_CPU = False #@param {type:"boolean"}
CPU_WORKERS = 2 #@param [1, 2] {type:"raw"}
CHECKPOINT_EVERY = 25 #@param {type:"integer"}
MAX_TRIALS_THIS_SESSION = 0 #@param {type:"integer"}
# MAX_TRIALS_THIS_SESSION = 0 означает без ограничения на число повторов в этом сеансе.
if START_CPU:
    subprocess.run([str(CPU_PY), str(CODE / 'fly_colab.py'), '--root', str(ROOT), '--mode', 'cpu', '--workers', str(CPU_WORKERS), '--checkpoint-every', str(CHECKPOINT_EVERY), '--max-trials', str(MAX_TRIALS_THIS_SESSION)], check=True)
else:
    print('Расчёт не запущен. Для старта включите START_CPU и выполните ячейку.')

## 6. Отдельный экспериментальный GPU-пилот

**Не часть полного плана.** Для него потребуется GPU runtime Colab, NVIDIA CUDA и установка отдельного окружения. Смена типа среды завершит текущий сеанс, поэтому сначала остановите CPU и дождитесь сохранения снимка. Вернитесь и выполните ячейки 1, 3 и 3б. Затем включите флажок ниже.

Будут выполнены контроль и одна лезия при 100 Гц, seed 0, сначала на CPU, затем на GPU. CPU-пилот не добавляется в полную серию, GPU не добавляется в CPU-базу. Brian2CUDA использует другую версию Brian2 и другой генератор случайных чисел, поэтому одинаковые seed не гарантируют одинаковые спайки. Времена CPU и GPU также не являются корректным сравнением скорости: компиляция GPU включена, а время CPU из кода 28 не включает построение сети. Для выводов понадобится расширенная проверка и измерение. Проба может не запуститься или не поместиться в память бесплатной видеокарты. В этом случае CPU-серия остаётся независимой.

GPU-путь не проверен нами на настоящем Colab/GPU. Если будет ошибка, сохраните текст ячейки и пришлите его; не запускайте полный план на неподтверждённом GPU-коде.

In [ ]:
#@title 6. Попробовать CPU и GPU отдельно
RUN_GPU_PILOT = False #@param {type:"boolean"}
if RUN_GPU_PILOT:
    import shutil
    if not shutil.which('nvidia-smi') or not shutil.which('nvcc'):
        raise RuntimeError('Нужен GPU runtime Colab с NVIDIA GPU и nvcc. GPU-пилот не запущен.')
    subprocess.run(['nvidia-smi'], check=True)
    GPU_PY = Path('/content/fly-shiu-gpu/bin/python')
    if not GPU_PY.exists():
        subprocess.run([sys.executable, '-m', 'venv', str(GPU_PY.parents[1])], check=True)
    subprocess.run([str(GPU_PY), '-m', 'pip', 'install', '-q', 'numpy==2.2.6', 'pandas==2.2.3', 'pyarrow==18.1.0', 'brian2cuda==1.0b1', 'psutil==6.1.1'], check=True)
    subprocess.run([str(CPU_PY), str(CODE / 'fly_colab.py'), '--root', str(ROOT), '--mode', 'cpu-pilot'], check=True)
    subprocess.run([str(GPU_PY), str(CODE / 'fly_colab.py'), '--root', str(ROOT), '--mode', 'gpu-pilot'], check=True)
    print('Пилоты сохранены отдельно в 04_Отдельные_GPU_пробы. Они не входят в полный план.')
else:
    print('GPU-пилот не запущен.')

## 7. Экспорт для приложения

При необходимости включите флажок ниже. Будет проверен последний снимок CPU-серии и создана **новая** папка с `lesions.sqlite3` на Диске. Её можно скачать после завершения записи. Для обзора в приложении потребуется также папка того же топ-200. Автоматическое продолжение этого файла на Windows может быть запрещено из-за различий версий Python; обзор не запускает симуляцию. Экспорт не изменяет исходные снимки.

In [ ]:
#@title 7. Создать проверенную копию SQLite для приложения
EXPORT_CPU = False #@param {type:"boolean"}
if EXPORT_CPU:
    subprocess.run([str(CPU_PY), str(CODE / 'fly_colab.py'), '--root', str(ROOT), '--mode', 'export'], check=True)
else:
    print('Экспорт не запущен.')